# Domain-Specific Text Analysis and Retrieval System
## Domain: Employment and Work-Related Contracts (India)

**Course:** NLP, Assessment 1, Vidyashilp University. **Deadline:** 04.10.2026

This notebook implements the full pipeline required by the assignment (Modules 1 to 6, Exercises 1 to 14) for employment and work-related contracts: employment agreements, offer letters, bonds, NDAs, non-compete and non-solicitation agreements, remote-work, contractor, consultancy, internship, part-time and gig-worker agreements.

**Guiding principle (Select, Order, Implement, Compare, Evaluate, Justify).** Every process decision below is driven by one domain fact: in a contract, *who* is bound and *whether* they are bound are the whole point. "The Employee **shall not** disclose" and "the Employee shall disclose" differ by one word. So this pipeline protects negations, modal verbs, party names, money, durations and statute references, and every comparison is designed to show why.

### How to run
1. Put 15 to 30 contracts in `./data/` (PDF, TXT, DOCX, HTML, JSON, CSV, or scanned images PNG/JPG/TIFF). IDs `D01, D02, ...` are assigned automatically in filename order. **Scanned PDFs are OCR'd automatically, page by page** (see Module 1 for the one-time OCR install).
2. Run all cells top to bottom. Every results CSV is written to `./results/`.
3. Fill in the three annotation files described in **"What you must do"** below, then re-run from Module 2 onward. Several evaluation numbers are only valid after this.

> **Cells marked `YOUR INPUT` need your judgement.** They contain seed values so the notebook runs end to end, but the seed values are illustrative. Replace them with your own annotations before reporting numbers.

In [26]:
# Setup. Uncomment the install line on a fresh environment.

# Install required dependencies
#%pip install -U rapidocr onnxruntime pillow pymupdf python-docx
# Note: the package is called `rapidocr`. The older `rapidocr-onnxruntime` is a frozen legacy line.

import os, re, json, time, glob, math, unicodedata, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
import spacy
from spacy.language import Language
import fitz as pymupdf               # PyMuPDF is imported using 'fitz'
import docx                         # python-docx
from bs4 import BeautifulSoup
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

for name, mod in [("spacy", spacy), ("nltk", nltk), ("pandas", pd), ("pymupdf", pymupdf)]:
    print(f"{name:10s} {mod.__version__}")
import tokenizers, sklearn
print(f"{'tokenizers':10s} {tokenizers.__version__}\n{'sklearn':10s} {sklearn.__version__}")

spacy      3.8.16
nltk       3.9.1
pandas     2.2.3
pymupdf    1.28.2
tokenizers 0.23.2
sklearn    1.6.1


In [27]:
# NLTK resources: these are the CURRENT resource names.
# "punkt" -> "punkt_tab" and "averaged_perceptron_tagger" -> "averaged_perceptron_tagger_eng" in NLTK 3.9+.
for res in ["punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger_eng", "treebank", "universal_tagset"]:
    nltk.download(res, quiet=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
nlp.max_length = 3_000_000
print("spaCy model:", nlp.meta["name"], nlp.meta["version"], "| pipes:", nlp.pipe_names)

spaCy model: core_web_sm 3.8.0 | pipes: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [28]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path inside Google Drive
DRIVE_BASE_DIR = "/content/drive/MyDrive/NLP_Assessment1"

CONFIG = {
    "DATA_DIR": os.path.join(DRIVE_BASE_DIR, "data"),
    "RESULTS_DIR": os.path.join(DRIVE_BASE_DIR, "results"),
    "ANNOTATION_DIR": os.path.join(DRIVE_BASE_DIR, "annotations"),     # your hand-made gold files live here
    "BPE_VOCAB_SIZE": 2000,                # raise to 5000+ if your corpus is large
    "TOP_K": 5,                            # K for Precision@K / Recall@K
    "MIN_DOCS_WARNING": 15,
    # ---- OCR for scanned documents (RapidOCR) ----
    "OCR_ENABLED": True,                   # False = never OCR (scanned pages are left empty)
    "OCR_DPI": 200,                        # resolution PDF pages are rendered at before OCR (200 is a good default; 300 is slower)
    "OCR_MIN_LONG_SIDE": 2300,             # IMAGE files smaller than this (pixels, long side) are enlarged first; 2300 = A4 at ~200 dpi
    "OCR_MIN_CHARS_PER_PAGE": 25,          # a page with fewer text-layer characters AND an image is treated as a scan
    "OCR_WARN_CONF": 85,                   # warn when a document's mean OCR confidence (0-100) is below this
    "OCR_RESTORE_RUPEE": "safe",           # "off", "safe" (restore only unmistakable cases) or "context" (also amounts next to money words)
}

for d in ["DATA_DIR", "RESULTS_DIR", "ANNOTATION_DIR"]:
    os.makedirs(CONFIG[d], exist_ok=True)

def save(df, name):
    """Write a results table to the Google Drive results folder and return it for display."""
    path = os.path.join(CONFIG["RESULTS_DIR"], name)
    df.to_csv(path, index=False)
    print(f"  saved -> {path}")
    return df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
# Module 1: Document Collection and Extraction

**Why first:** every later stage consumes this text, so extraction errors (merged words, lost currency symbols, page headers) propagate everywhere. Cleaning is deliberately *conservative*: it removes layout noise (page numbers, running headers, and line-leading enumeration labels such as `1.` or `(a)`, which are layout rather than content) but keeps in-text clause references like `Clause 4.2(a)`, currency symbols, punctuation and case, because later stages (custom tokenizer, NER, POS) need them. Lowercasing and punctuation stripping happen later, inside tokenization, where they can be compared.

## Scanned documents and OCR

A scanned PDF has no text layer, only a picture of each page, so ordinary extraction returns nothing and every later stage silently sees an empty document. This notebook therefore runs **OCR (optical character recognition)** with **RapidOCR** where needed. Design decisions:

- **Per page, not per document.** A page with almost no text layer *and* an embedded image is treated as a scan. Mixed PDFs (typed pages plus a scanned signature page or annexure) work, and typed pages are never OCR'd, since OCR is slower and less accurate than a real text layer. Image files (PNG, JPG, TIFF, BMP) are OCR'd directly.
- **Why RapidOCR.** It installs with `pip` alone (models are bundled, no separate program and no internet needed at run time) and runs on the CPU through ONNX Runtime. The engine is loaded only when the first scanned page is found, so a corpus of typed documents pays nothing. Expect a few seconds per page on an ordinary CPU.
- **Measured weaknesses on Indian contracts, which the steps below are built around.**
  1. **It often drops the rupee sign.** In a test of 60 rendered amounts at ordinary body-text size it kept the `₹` only 23 times (38%) and silently dropped it the other 37, so `₹12,00,000` became `12,00,000`. Large text fared better, which is why a quick test on a big sample can look reassuring. It occasionally reads `₹` as a yen sign.
  2. **It can lose the space between words in all-caps headings** (`EMPLOYMENT AGREEMENT` read as `EMPLOYMENTAGREEMENT`). Body text was unaffected in testing. Headings are a small share of tokens, but check them.
  3. **Low resolution loses whole lines.** The same page read at about 150 dpi lost the line naming the parties and garbled the notice-period clause into noise; at 187 dpi and above it was read perfectly. PDF pages are re-rendered at `OCR_DPI`, so they are safe. **Image files are enlarged to `OCR_MIN_LONG_SIDE` (about A4 at 200 dpi) before OCR**; they are never shrunk. A genuinely low-quality photo can still fail, and enlarging cannot add detail that is not there.
  4. **Its confidence score is not a safety net.** It stayed at 96 to 99 in every test, including the runs that dropped a line or lost a rupee sign. A high score does not show the text is right, which is why the report below lists every amount for you to check against the page.
- **A narrow, reported repair for the rupee sign.** A dropped sign cannot be known for certain, so the repair only acts where the text leaves almost no doubt, and every change is counted in the OCR report:
  - a yen sign directly before a digit becomes `₹` (the sign never appears in an Indian contract);
  - an amount ending in `/-` with no currency marker gets `₹` (that suffix is an Indian rupee convention);
  - an amount in lakh grouping (`12,00,000`, `5,00,000`) with no marker gets `₹`, unless a unit follows (`1,00,000 shares`).

  Western-grouped amounts such as `15,000` are ambiguous (a stipend or a count?). In the default `safe` mode they are **not** changed, only counted as *unresolved* when a money word such as `stipend` or `salary` sits nearby. Setting `CONFIG["OCR_RESTORE_RUPEE"] = "context"` restores those too, at the risk of a few false positives. `"off"` disables all repairs.
- **English only.** The configured models read English. Hindi or other Indic scripts would need a different recognition model, which has not been set up or tested here.
- **Provenance is recorded.** `document_map.csv` gains columns for OCR pages, engine version, mean confidence and repairs, so the report can state exactly which documents were OCR'd.

Cleaning below also copes with OCR's hard-wrapped lines (it rejoins wrapped lines into paragraphs so sentence detection works) and keeps the hyphen in words such as `non-compete` when a line break falls inside them.

In [29]:
# ---------------- OCR for scanned documents (RapidOCR) ----------------
from importlib.metadata import version as _pkg_version, PackageNotFoundError
from PIL import Image, ImageSequence

# --- Repairing the rupee sign (see the Module 1 notes for the evidence behind these rules) ---
CURRENCY_BEFORE = re.compile(r"(?:₹|Rs\.?|INR|Re\.?|\$|USD|€|£)\s*$", re.IGNORECASE)
UNIT_AFTER = re.compile(r"\s*(?:shares?|units?|employees?|hours?|days?|months?|years?|sq|square|kg|km|copies|pages|words|%)\b",
                        re.IGNORECASE)
YEN_MISREAD = re.compile(r"[¥￥](?=\s?\d)")
SLASH_DASH_NUM = re.compile(r"(?<![\d,.])\d[\d,]*(?:\.\d+)?(?=/-)")                      # 5,00,000/-
LAKH_NUM = re.compile(r"(?<![\d,.])\d{1,2}(?:,\d{2})+,\d{3}(?:\.\d+)?(?!\d)(?!,\d)")      # 12,00,000  1,20,00,000
WESTERN_NUM = re.compile(r"(?<![\d,.])\d{1,3}(?:,\d{3})+(?:\.\d+)?(?!\d)(?!,\d)")          # 15,000  (ambiguous)
MONEY_CUES = re.compile(r"\b(?:stipend|salary|ctc|fees?|penalty|penalties|bonus|compensation|remuneration|amount|"
                        r"sum|payable|pay|paid|rent|deposit|costs?|damages|rupees)\b", re.IGNORECASE)

def repair_ocr_text(text, mode=None):
    # Returns (repaired_text, number_of_rupee_signs_added_or_fixed, number_of_unresolved_candidates).
    mode = (mode or CONFIG["OCR_RESTORE_RUPEE"]).lower()
    if mode == "off":
        return text, 0, 0
    text, n_yen = YEN_MISREAD.subn("₹", text)
    has_marker = lambda pos: bool(CURRENCY_BEFORE.search(text[max(0, pos - 6):pos]))
    inserts, unresolved = set(), 0
    for m in SLASH_DASH_NUM.finditer(text):
        if not has_marker(m.start()):
            inserts.add(m.start())
    for m in LAKH_NUM.finditer(text):
        if not has_marker(m.start()) and not UNIT_AFTER.match(text, m.end()):
            inserts.add(m.start())
    for m in WESTERN_NUM.finditer(text):
        if m.start() in inserts or has_marker(m.start()) or UNIT_AFTER.match(text, m.end()):
            continue                     # already restored by an earlier rule, already marked, or not money
        if MONEY_CUES.search(text[max(0, m.start() - 40):m.start()]):
            if mode == "context":
                inserts.add(m.start())
            else:
                unresolved += 1
    for pos in sorted(inserts, reverse=True):
        text = text[:pos] + "₹" + text[pos:]
    return text, n_yen + len(inserts), unresolved

# --- RapidOCR engine: loaded once, lazily ---
_OCR = {"tried": False, "engine": None, "version": ""}

def get_ocr():
    if _OCR["tried"]:
        return _OCR["engine"]
    _OCR["tried"] = True
    if not CONFIG["OCR_ENABLED"]:
        print("OCR is switched off (CONFIG['OCR_ENABLED'] = False): scanned pages will be left empty.")
        return None
    try:
        from rapidocr import RapidOCR
        _OCR["engine"] = RapidOCR(params={"Global.log_level": "warning"})
        try:
            _OCR["version"] = _pkg_version("rapidocr")
        except PackageNotFoundError:
            pass
        print(f"OCR engine: RapidOCR {_OCR['version']} (loaded now because a scanned page was found)")
    except Exception as e:
        print(f"!! RapidOCR could not be loaded ({type(e).__name__}: {str(e)[:100]}).")
        print("   Scanned pages will be EMPTY. Fix:  pip install -U rapidocr onnxruntime")
    return _OCR["engine"]

def _ocr_lines(img, engine):
    # Run RapidOCR and rebuild reading order: group text boxes that share a line, then read top to bottom.
    res = engine(np.asarray(img.convert("RGB")))
    if res is None or res.txts is None or len(res.txts) == 0:
        return "", 0.0
    items = []
    for box, txt, sc in zip(res.boxes, res.txts, res.scores):
        ys = [p[1] for p in box]; xs = [p[0] for p in box]
        items.append((float(np.mean(ys)), min(xs), max(ys) - min(ys), txt, float(sc)))
    items.sort(key=lambda t: t[0])
    lines, cur, cur_y, cur_h = [], [], None, 0.0
    for y, x, h, txt, sc in items:
        if cur_y is None or abs(y - cur_y) <= 0.6 * max(cur_h, h):
            cur.append((x, txt)); cur_y = y if cur_y is None else (cur_y + y) / 2; cur_h = max(cur_h, h)
        else:
            lines.append(" ".join(t for _, t in sorted(cur))); cur, cur_y, cur_h = [(x, txt)], y, h
    if cur:
        lines.append(" ".join(t for _, t in sorted(cur)))
    return "\n".join(lines), 100 * float(np.mean([i[4] for i in items]))

def ocr_image(img):
    # OCR one PIL image. Returns (text, mean_confidence_0_to_100, rupee_repairs, unresolved_amounts).
    engine = get_ocr()
    if engine is None:
        return "", 0.0, 0, 0
    try:
        text, conf = _ocr_lines(img, engine)
    except Exception as e:
        print(f"  !! OCR failed on one page ({type(e).__name__}: {str(e)[:80]}); page left empty.")
        return "", 0.0, 0, 0
    text, repairs, unresolved = repair_ocr_text(text)
    return text, conf, repairs, unresolved

def page_to_image(page):
    pix = page.get_pixmap(dpi=CONFIG["OCR_DPI"], colorspace=pymupdf.csGRAY)
    return Image.frombytes("L", (pix.width, pix.height), pix.samples)

def enlarge_for_ocr(img):
    # Image files arrive at whatever resolution they have (PDF pages are re-rendered at OCR_DPI instead).
    # Small images lose or garble whole lines, so enlarge them to the minimum size; never shrink.
    long_side = max(img.size)
    if long_side >= CONFIG["OCR_MIN_LONG_SIDE"]:
        return img, 1.0
    scale = CONFIG["OCR_MIN_LONG_SIDE"] / long_side
    return img.resize((round(img.width * scale), round(img.height * scale)), Image.LANCZOS), scale

OCR_LOG = {}     # path -> what OCR did for that file (feeds document_map.csv and the OCR report)

def _log(path, pages, ocr_pages, unread, confs, repairs, unresolved, first):
    OCR_LOG[path] = {"pages": pages, "ocr_pages": ocr_pages, "unread_scanned_pages": unread,
                     "ocr_engine": f"rapidocr {_OCR['version']}".strip() if ocr_pages else "",
                     "ocr_mean_conf": round(float(np.mean(confs)), 1) if confs else np.nan,
                     "ocr_repairs": repairs, "ocr_unresolved_amounts": unresolved, "first_ocr_page": first}

print(f"OCR configured: enabled={CONFIG['OCR_ENABLED']}, dpi={CONFIG['OCR_DPI']}, "
      f"rupee repair={CONFIG['OCR_RESTORE_RUPEE']!r} (RapidOCR is loaded only if a scanned page is found)")

OCR configured: enabled=True, dpi=200, rupee repair='safe' (RapidOCR is loaded only if a scanned page is found)


In [30]:
SUPPORTED = (".pdf", ".txt", ".docx", ".html", ".htm", ".json", ".csv",
             ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

def read_pdf(path):
    # Per PAGE: a page with almost no text layer but an embedded image is treated as a scan and OCR'd.
    # Mixed documents (some typed pages, some scanned) therefore work, and typed pages are never OCR'd.
    parts, n_ocr, n_unread, confs, repairs, unresolved, first = [], 0, 0, [], 0, 0, None
    with pymupdf.open(path) as doc:
        n_pages = doc.page_count
        for pno, page in enumerate(doc):
            txt = page.get_text()
            if len(txt.strip()) < CONFIG["OCR_MIN_CHARS_PER_PAGE"] and page.get_images():
                if get_ocr() is None:
                    n_unread += 1
                else:
                    t0 = time.perf_counter()
                    txt, conf, r, u = ocr_image(page_to_image(page))
                    print(f"  OCR {os.path.basename(path)} page {pno + 1}/{n_pages}: "
                          f"{time.perf_counter() - t0:.1f}s, confidence {conf:.0f}")
                    n_ocr += 1; confs.append(conf); repairs += r; unresolved += u
                    first = pno if first is None else first
            parts.append(txt)
    _log(path, n_pages, n_ocr, n_unread, confs, repairs, unresolved, first)
    return "\n".join(parts)

def read_image(path):
    # Scanned images (PNG, JPG, TIFF, BMP). Multi-page TIFFs are read frame by frame.
    with Image.open(path) as im:
        frames = [f.convert("L").copy() for f in ImageSequence.Iterator(im)]
    parts, n_ocr, n_unread, confs, repairs, unresolved = [], 0, 0, [], 0, 0
    for k, f in enumerate(frames, start=1):
        if get_ocr() is None:
            n_unread += 1
        else:
            t0 = time.perf_counter()
            f, scale = enlarge_for_ocr(f)
            txt, conf, r, u = ocr_image(f)
            print(f"  OCR {os.path.basename(path)} frame {k}/{len(frames)}: "
                  f"{time.perf_counter() - t0:.1f}s, confidence {conf:.0f}"
                  + (f", enlarged x{scale:.2f} first" if scale > 1 else ""))
            parts.append(txt); n_ocr += 1; confs.append(conf); repairs += r; unresolved += u
    _log(path, len(frames), n_ocr, n_unread, confs, repairs, unresolved, 0 if n_ocr else None)
    return "\n".join(parts)

def read_docx(path):
    return "\n".join(p.text for p in docx.Document(path).paragraphs)

def read_html(path):
    with open(path, encoding="utf-8", errors="ignore") as f:
        return BeautifulSoup(f.read(), "html.parser").get_text(separator="\n")

def read_json(path):
    with open(path, encoding="utf-8", errors="ignore") as f:
        data = json.load(f)
    def walk(x):
        if isinstance(x, str): return [x]
        if isinstance(x, dict): return [s for v in x.values() for s in walk(v)]
        if isinstance(x, list): return [s for v in x for s in walk(v)]
        return []
    return "\n".join(walk(data))

def read_csv(path):
    df = pd.read_csv(path, dtype=str).fillna("")
    return "\n".join(" ".join(row) for row in df.values)

def read_txt(path):
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read()

READERS = {".pdf": read_pdf, ".docx": read_docx, ".html": read_html, ".htm": read_html,
           ".json": read_json, ".csv": read_csv, ".txt": read_txt,
           ".png": read_image, ".jpg": read_image, ".jpeg": read_image,
           ".tif": read_image, ".tiff": read_image, ".bmp": read_image}

files = sorted(p for p in glob.glob(os.path.join(CONFIG["DATA_DIR"], "*"))
               if p.lower().endswith(SUPPORTED))
if not files:
    raise SystemExit(f"No documents found in {CONFIG['DATA_DIR']}. Add 15 to 30 contracts and re-run.")

raw_docs, doc_meta = {}, []
for i, path in enumerate(files, start=1):
    did = f"D{i:02d}"
    ext = os.path.splitext(path)[1].lower()
    try:
        text = READERS[ext](path)
    except Exception as e:
        print(f"  !! {os.path.basename(path)}: could not read ({type(e).__name__}: {e})")
        continue
    log = OCR_LOG.get(path, {})
    if len(text.strip()) < 200:
        why = ("scanned pages were left unread because no OCR backend is available"
               if log.get("unread_scanned_pages") else "very little extractable text")
        print(f"  !! {did} {os.path.basename(path)}: only {len(text.strip())} chars ({why}). Kept, but check it.")
    raw_docs[did] = text
    doc_meta.append({"doc_id": did, "file": os.path.basename(path), "format": ext[1:],
                     "pages": log.get("pages"), "ocr_pages": log.get("ocr_pages", 0),
                     "unread_scanned_pages": log.get("unread_scanned_pages", 0),
                     "ocr_engine": log.get("ocr_engine", ""), "ocr_mean_conf": log.get("ocr_mean_conf", np.nan),
                     "ocr_repairs": log.get("ocr_repairs", 0),
                     "ocr_unresolved_amounts": log.get("ocr_unresolved_amounts", 0)})

doc_map = pd.DataFrame(doc_meta)
save(doc_map, "document_map.csv")
if len(raw_docs) < CONFIG["MIN_DOCS_WARNING"]:
    print(f"\nWARNING: {len(raw_docs)} documents loaded; the assignment requires at least 15.")
doc_map

OCR engine: RapidOCR 3.9.2 (loaded now because a scanned page was found)
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 6/289: 15.9s, confidence 97
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 9/289: 18.1s, confidence 97
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 16/289: 19.0s, confidence 94
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 29/289: 15.4s, confidence 95
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 32/289: 18.9s, confidence 96
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 39/289: 15.7s, confidence 94
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 62/289: 13.5s, confidence 95
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 65/289: 15.6s, confidence 96
  OCR 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf page 72/2

,doc_id,file,format,pages,ocr_pages,unread_scanned_pages,ocr_engine,ocr_mean_conf,ocr_repairs,ocr_unresolved_amounts
0,D01,01_Gintaa_Gig_Worker_Delivery_Partner_Terms.url.txt,txt,NaN,0,0,,NaN,0,0
1,D02,01_IndusInd_Nippon_Life_Part_Time_Appointment.url.txt,txt,NaN,0,0,,NaN,0,0
2,D03,01_Payoneer_Independent_Contractor_Agreement.url.txt,txt,NaN,0,0,,NaN,0,0
3,D04,01_Sears_IT_Internship_Agreement.url.txt,txt,NaN,0,0,,NaN,0,0
4,D05,01_Viasat_India_Remote_Work_Agreement.url.txt,txt,NaN,0,0,,NaN,0,0
5,D06,01_i-Venture_Consultancy_Agreement.url.txt,txt,NaN,0,0,,NaN,0,0
6,D07,02_AERA_Consultancy_Agreement.url.txt,txt,NaN,0,0,,NaN,0,0
7,D08,02_Doxuno_India_Independent_Contractor_Agreement.url.txt,txt,NaN,0,0,,NaN,0,0
8,D09,02_Jaitra_Internship_Agreement.url.txt,txt,NaN,0,0,,NaN,0,0
9,D10,02_MealVersity_Gig_Worker_Delivery_Partner_Terms.url.txt,txt,NaN,0,0,,NaN,0,0


In [22]:
# Print the exact Google Drive directory path where contracts must be uploaded
print("Please upload your 15 to 30 contract files to this exact directory in Google Drive:")
print(CONFIG["DATA_DIR"])


Please upload your 15 to 30 contract files to this exact directory in Google Drive:
/content/drive/MyDrive/NLP_Assessment1/data


In [24]:
import os

data_dir = CONFIG["DATA_DIR"]
if os.path.exists(data_dir):
    files_in_dir = os.listdir(data_dir)
    print(f"Found {len(files_in_dir)} item(s) in {data_dir}:")
    for item in sorted(files_in_dir):
        print(f" - {item}")
else:
    print(f"Directory does not exist: {data_dir}")

Found 44 item(s) in /content/drive/MyDrive/NLP_Assessment1/data:
 - 01_Gintaa_Gig_Worker_Delivery_Partner_Terms.url.txt
 - 01_IndusInd_Nippon_Life_Part_Time_Appointment.url.txt
 - 01_Payoneer_Independent_Contractor_Agreement.url.txt
 - 01_Sears_IT_Internship_Agreement.url.txt
 - 01_Viasat_India_Remote_Work_Agreement.url.txt
 - 01_i-Venture_Consultancy_Agreement.url.txt
 - 02_AERA_Consultancy_Agreement.url.txt
 - 02_Doxuno_India_Independent_Contractor_Agreement.url.txt
 - 02_Jaitra_Internship_Agreement.url.txt
 - 02_MealVersity_Gig_Worker_Delivery_Partner_Terms.url.txt
 - 02_Reliance_Jio_Part_Time_Appointment_2023.url.txt
 - 02_WonderLegal_India_Remote_Work_Agreement.url.txt
 - 1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf
 - 1714631710.pdf
 - 17659708753175.pdf
 - 2018-19-placed-student-details-Civil.pdf
 - 613578483-Remote-Agreement-Pulkit-Garg-47.pdf
 - AIM (25-11-2023 09_53_38pm) - Sineka N offer letter.pdf
 - Agreement Contracts.pdf
 - Basic Non-Disclosure Agree

In [31]:
# OCR quality report. Only shows anything if at least one scanned page was found.
ocr_docs = doc_map[doc_map.ocr_pages > 0]
unread = doc_map[doc_map.unread_scanned_pages > 0]

if len(unread):
    print(f"!! {int(unread.unread_scanned_pages.sum())} scanned page(s) in {len(unread)} document(s) were NOT read "
          "because OCR is unavailable or switched off. Those documents are INCOMPLETE:")
    display(unread[["doc_id", "file", "pages", "unread_scanned_pages"]])

if ocr_docs.empty:
    print("No pages were OCR'd." if len(unread) else "No scanned pages found, so OCR was not needed.")
else:
    print(f"OCR was used on {int(ocr_docs.ocr_pages.sum())} page(s) across {len(ocr_docs)} document(s).\n")
    cols = ["doc_id", "file", "pages", "ocr_pages", "ocr_engine", "ocr_mean_conf", "ocr_repairs", "ocr_unresolved_amounts"]
    display(ocr_docs[cols])
    save(ocr_docs[cols], "ocr_report.csv")
    low = ocr_docs[ocr_docs.ocr_mean_conf < CONFIG["OCR_WARN_CONF"]]
    if len(low):
        print(f"!! Mean OCR confidence is below {CONFIG['OCR_WARN_CONF']} for: {', '.join(low.doc_id)}. "
              "Expect word errors; try a better scan or a higher CONFIG['OCR_DPI'].")
    if ocr_docs.ocr_repairs.sum():
        print(f"{int(ocr_docs.ocr_repairs.sum())} rupee sign(s) were restored or corrected automatically "
              f"(mode {CONFIG['OCR_RESTORE_RUPEE']!r}); RapidOCR often drops the sign at body-text size.")
    if ocr_docs.ocr_unresolved_amounts.sum():
        print(f"{int(ocr_docs.ocr_unresolved_amounts.sum())} amount(s) next to money words have NO currency marker and "
              "were left unchanged (ambiguous, e.g. '15,000'). Check them by hand, or set "
              "CONFIG['OCR_RESTORE_RUPEE'] = 'context' to restore them.")
    print("\nEvery amount found in OCR'd text. Compare each against the page image:")
    AMOUNT_LINE = re.compile(r"\d{1,3}(?:,\d{2,3})+|\bRs\.?\s?\d|INR\s?\d|₹")
    shown = 0
    for _, r in ocr_docs.iterrows():
        for line in raw_docs[r.doc_id].split("\n"):
            if AMOUNT_LINE.search(line) and shown < 12:
                print(f"  [{r.doc_id}] {line.strip()[:110]}"); shown += 1
    print("\nCaution: confidence stayed at 96 to 99 in testing even when a line was dropped or a rupee sign lost,")
    print("so a high score does not prove the text is right. Read the amounts above against the page image.")
    print("OCR errors flow into every later number in this notebook. State that in your report's limitations.")

OCR was used on 315 page(s) across 7 document(s).



,doc_id,file,pages,ocr_pages,ocr_engine,ocr_mean_conf,ocr_repairs,ocr_unresolved_amounts
12,D13,1669358273-Offer_Appointment_letters_required_by_NAAC_for_DVV.pdf,289.0,28,rapidocr 3.9.2,96.0,0,5
14,D15,17659708753175.pdf,48.0,3,rapidocr 3.9.2,96.5,0,0
18,D19,Agreement Contracts.pdf,61.0,6,rapidocr 3.9.2,98.4,0,0
24,D25,DVV-3.5.2.1.pdf,295.0,226,rapidocr 3.9.2,96.8,3,2
40,D41,aliens group offer letters.pdf,109.0,31,rapidocr 3.9.2,79.8,0,0
41,D42,ceo-executive-employment-agreement2022.pdf,19.0,19,rapidocr 3.9.2,98.2,0,0
43,D44,service-provider-agreement.pdf,21.0,2,rapidocr 3.9.2,98.6,0,0


  saved -> /content/drive/MyDrive/NLP_Assessment1/results/ocr_report.csv
!! Mean OCR confidence is below 85 for: D41. Expect word errors; try a better scan or a higher CONFIG['OCR_DPI'].
3 rupee sign(s) were restored or corrected automatically (mode 'safe'); RapidOCR often drops the sign at body-text size.
7 amount(s) next to money words have NO currency marker and were left unchanged (ambiguous, e.g. '15,000'). Check them by hand, or set CONFIG['OCR_RESTORE_RUPEE'] = 'context' to restore them.

Every amount found in OCR'd text. Compare each against the page image:
  [D13] 4,76,214/- p.a. (Rupees Four Lakhs Seventy Six Thousand Two Hundred and Fourteen only per
  [D13] 650,400
  [D13] be liable to pay to the Company liquidated damages of up to Rs. 75,000/-(Rupees Seventy Five Thousand only) in
  [D13] b. Group Term Life Insurance: Rs. 14, 00,000 in the unfortunate event of death on account of either accidents 
  [D13] annum to avoid perquisite tax (perquisite tax is applicable on contr

In [32]:
KEEP_HYPHEN = {"non", "self", "co", "ex", "anti", "work", "part", "full", "year", "long", "short",
               "third", "cross", "on", "off", "in", "out", "pre", "post", "sub", "inter", "multi"}
ABBR_END = re.compile(r"\b(?:Pvt|Ltd|Co|Inc|No|Rs|Sr|Jr|Dr|Mr|Mrs|Ms|viz|Govt|Dept)\.$")

def _dehyphenate(m):
    # "termi-\nnation" -> "termination", but "non-\ncompete" -> "non-compete" (a real compound).
    return f"{m.group(1)}-{m.group(2)}" if m.group(1).lower() in KEEP_HYPHEN else f"{m.group(1)}{m.group(2)}"

def rejoin_wrapped_lines(lines):
    """Join hard-wrapped lines (common in PDFs, universal in OCR output) back into paragraphs.
    A line is joined to the next when it does not end a sentence AND either the next line starts
    lowercase or this physical line runs close to the document's typical line width."""
    lens = sorted(len(l) for l in lines if len(l) > 20)
    width = lens[int(0.9 * (len(lens) - 1))] if lens else 0
    out, last_len = [], 0
    for l in lines:
        if out and out[-1] and l:
            prev = out[-1]
            ended = bool(re.search(r"[.:;!?]$", prev)) and not ABBR_END.search(prev)
            if not ended and (l[0].islower() or last_len >= 0.8 * width):
                out[-1] = prev + " " + l; last_len = len(l)
                continue
        out.append(l); last_len = len(l)
    return out

def clean_text(text):
    """Conservative cleaning. Removes layout noise, keeps legal content."""
    text = unicodedata.normalize("NFKC", text)            # unify look-alike characters; keeps the rupee sign
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[\u2018\u2019]", "'", text)             # curly single quotes
    text = re.sub(r"[\u201c\u201d]", '"', text)             # curly double quotes
    text = re.sub(r"(\w+)-\n(\w)", _dehyphenate, text)     # re-join words hyphenated across a line break
    text = re.sub(r"(?im)^\s*page\s+\d+(\s+of\s+\d+)?\s*$", "", text)   # "Page 1 of 2"
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)            # bare page numbers
    text = re.sub(r"(?m)^\s*(?:\d+(?:\.\d+)*|[a-z]|[ivx]+)[.)]\s+", "", text)  # enumeration labels "1.", "(a)", "iv)"
    lines = [l.strip() for l in text.split("\n")]
    counts = Counter(l for l in lines if 0 < len(l) < 60)
    repeated = {l for l, c in counts.items() if c >= 3}    # running headers/footers repeated on many pages
    lines = [l for l in lines if l not in repeated]
    lines = rejoin_wrapped_lines(lines)
    text = "\n".join(lines)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

docs = {did: clean_text(t) for did, t in raw_docs.items()}
corpus = "\n".join(docs.values())

sample = next(iter(docs))
print(f"Sample {sample}, first 400 characters after cleaning:\n")
print(docs[sample][:400])

Sample D01, first 400 characters after cleaning:

Document: 01 Gintaa Gig Worker Delivery Partner Terms
Jurisdiction: India
Document type: Platform agreement
Source URL: https://gintaa.com/legal/delivery-partner/terms-condition
Use the source URL to access the original document. This file is a source pointer, not a reconstructed copy.


In [33]:
# Document statistics. spaCy's parser gives sentence boundaries; tokens here are spaCy tokens
# excluding pure whitespace. These are the "Before Processing" numbers used in Table A.
spacy_docs = {did: nlp(t) for did, t in docs.items()}

rows = []
for did, sd in spacy_docs.items():
    toks = [t.text for t in sd if not t.is_space]
    rows.append({"doc_id": did,
                 "sentences": sum(1 for _ in sd.sents),
                 "tokens": len(toks),
                 "characters": len(docs[did]),
                 "vocabulary": len(set(toks))})
stats = pd.DataFrame(rows)

all_tokens = [t.text for sd in spacy_docs.values() for t in sd if not t.is_space]
summary = pd.DataFrame([{
    "doc_id": "TOTAL",
    "sentences": stats.sentences.sum(),
    "tokens": stats.tokens.sum(),
    "characters": stats.characters.sum(),
    "vocabulary": len(set(all_tokens)),
}])
stats_out = pd.concat([stats, summary], ignore_index=True)
save(stats_out, "document_statistics.csv")

print(f"\nNumber of documents:     {len(docs)}")
print(f"Number of sentences:     {stats.sentences.sum():,}")
print(f"Number of tokens:        {stats.tokens.sum():,}")
print(f"Number of characters:    {stats.characters.sum():,}")
print(f"Vocabulary size:         {len(set(all_tokens)):,}")
print(f"Average document length: {stats.tokens.mean():,.1f} tokens")
stats_out

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/document_statistics.csv

Number of documents:     44
Number of sentences:     10,200
Number of tokens:        337,719
Number of characters:    1,855,282
Vocabulary size:         19,966
Average document length: 7,675.4 tokens


,doc_id,sentences,tokens,characters,vocabulary
0,D01,2,43,286,34
1,D02,2,44,319,34
2,D03,3,43,312,34
3,D04,3,41,280,32
4,D05,2,45,309,34
5,D06,3,41,284,32
6,D07,2,40,273,31
7,D08,2,43,300,33
8,D09,2,40,282,31
9,D10,2,43,294,34


---
# Module 2: Preprocessing

## Exercises 1 and 2: Built-in tokenizers and dictionary size

Two dictionaries are compared for each tokenizer:
- **Token-based dictionary:** every unique token string exactly as produced (so `Employee` and `employee` are separate entries, and punctuation counts).
- **Term-based dictionary:** unique *terms* after lowercasing and discarding punctuation-only tokens. This is closer to what an index actually stores.

The gap between the two shows how much of a raw dictionary is case duplication and punctuation noise.

In [35]:
from nltk.tokenize import word_tokenize, wordpunct_tokenize, RegexpTokenizer, sent_tokenize

regex_word = RegexpTokenizer(r"\w+")
BUILTIN = {
    "whitespace":       lambda t: t.split(),
    "nltk_word":        lambda t: word_tokenize(t),
    "nltk_wordpunct":   lambda t: wordpunct_tokenize(t),
    "nltk_regex_\\w+":  lambda t: regex_word.tokenize(t),
    "spacy":            lambda t: [tok.text for tok in nlp.tokenizer(t) if not tok.is_space],
}

def term_dictionary(tokens):
    return {t.lower() for t in tokens if re.search(r"\w", t)}

builtin_tokens, rows = {}, []
for name, fn in BUILTIN.items():
    toks = [tok for t in docs.values() for tok in fn(t)]
    builtin_tokens[name] = toks
    rows.append({"tokenizer": name, "tokens": len(toks),
                 "token_dictionary": len(set(toks)),
                 "term_dictionary": len(term_dictionary(toks)),
                 "reduction_%": round(100 * (1 - len(term_dictionary(toks)) / max(len(set(toks)), 1)), 1)})
ex1 = pd.DataFrame(rows)
ex1

,tokenizer,tokens,token_dictionary,term_dictionary,reduction_%
0,whitespace,288375,27643,23719,14.2
1,nltk_word,330892,20564,17052,17.1
2,nltk_wordpunct,349894,18412,14551,21.0
3,nltk_regex_\w+,298523,18113,14551,19.7
4,spacy,337719,19966,16451,17.6


In [36]:
# Exercise 2: inspect what the built-in tokenizers actually produce, to find what needs fixing.
def problems(tokens):
    return {
        "punctuation_only":     sum(1 for t in tokens if not re.search(r"\w", t)),
        "bare_currency_symbol": sum(1 for t in tokens if t in {"₹", "Rs", "Rs.", "INR"}),
        "hyphen_fragments":     sum(1 for t in tokens if t == "-"),
        "case_duplicates":      len({t for t in tokens if t.isalpha()}) - len({t.lower() for t in tokens if t.isalpha()}),
        "single_char_tokens":   sum(1 for t in tokens if len(t) == 1 and t.isalpha()),
    }

ex2 = pd.DataFrame({name: problems(toks) for name, toks in builtin_tokens.items()}).T
display(ex2)

probe = "The CTC is ₹12,00,000 per annum, subject to Section 27 and Clause 4.2(a); a non-compete applies w.e.f. 15 April 2026."
print("\nHow each built-in tokenizer handles one domain sentence:\n")
for name, fn in BUILTIN.items():
    print(f"{name:16s} {fn(probe)}")

,punctuation_only,bare_currency_symbol,hyphen_fragments,case_duplicates,single_char_tokens
whitespace,2331,293,451,2884,6548
nltk_word,41719,363,523,3294,7718
nltk_wordpunct,51371,482,2865,3536,10054
nltk_regex_\w+,0,470,0,3536,10054
spacy,41650,361,2234,3361,7802



How each built-in tokenizer handles one domain sentence:

whitespace       ['The', 'CTC', 'is', '₹12,00,000', 'per', 'annum,', 'subject', 'to', 'Section', '27', 'and', 'Clause', '4.2(a);', 'a', 'non-compete', 'applies', 'w.e.f.', '15', 'April', '2026.']
nltk_word        ['The', 'CTC', 'is', '₹12,00,000', 'per', 'annum', ',', 'subject', 'to', 'Section', '27', 'and', 'Clause', '4.2', '(', 'a', ')', ';', 'a', 'non-compete', 'applies', 'w.e.f', '.', '15', 'April', '2026', '.']
nltk_wordpunct   ['The', 'CTC', 'is', '₹', '12', ',', '00', ',', '000', 'per', 'annum', ',', 'subject', 'to', 'Section', '27', 'and', 'Clause', '4', '.', '2', '(', 'a', ');', 'a', 'non', '-', 'compete', 'applies', 'w', '.', 'e', '.', 'f', '.', '15', 'April', '2026', '.']
nltk_regex_\w+   ['The', 'CTC', 'is', '12', '00', '000', 'per', 'annum', 'subject', 'to', 'Section', '27', 'and', 'Clause', '4', '2', 'a', 'a', 'non', 'compete', 'applies', 'w', 'e', 'f', '15', 'April', '2026']
spacy            ['The', 'CTC', 'is', 

**Observations to write up (Exercise 2).** Every built-in tokenizer fails on the same domain constructs, and each failure suggests a process:

| Problem seen | Consequence | Process that fixes it |
|---|---|---|
| `₹12,00,000` split into `₹` + `12,00,000` (or worse, `12` `,` `00` `,` `000`) | Salary figures become meaningless fragments | Custom tokenizer rule for Indian currency |
| `non-compete` split into `non` `-` `compete` | The clause name disappears; `compete` alone reverses meaning | Custom rule for hyphenated compounds |
| `Section 27`, `Clause 4.2(a)` split up | Statute and cross-references lost | Custom rules for legal references |
| `w.e.f.`, `Pvt.`, `Ltd.` split or mis-sentenced | Abbreviations treated as sentence ends | Abbreviation list |
| `Employee` vs `employee` | Case duplicates inflate the dictionary | Lowercasing (but only after NER, which needs case) |
| Punctuation-only tokens | Dictionary noise | Filter at the indexing stage, not at tokenization |

The `whitespace` tokenizer keeps `₹12,00,000` intact by accident but glues punctuation onto words (`annum,`), so it is not a real fix.

## Exercise 3: Custom and Hybrid tokenizers

**Domain tokenization problems (at least five required):**

| # | Problem | Example | Required behaviour |
|---|---|---|---|
| 1 | Indian currency formats | `₹12,00,000`, `Rs. 8,50,000/-`, `INR 15 LPA` | One token, including the lakh-style comma grouping |
| 2 | Hyphenated legal compounds | `non-compete`, `non-solicitation`, `work-from-home` | One token |
| 3 | Statute references | `Section 27`, `Indian Contract Act, 1872` | One token |
| 4 | Clause cross-references | `Clause 4.2(a)`, `4.2(a)` | One token |
| 5 | Abbreviations with periods | `Pvt.`, `Ltd.`, `w.e.f.`, `B.Tech` | One token, not a sentence break |
| 6 | Dates and times | `01/04/2026`, `9 a.m.` | One token |

**Custom tokenizer:** an ordered regular-expression tokenizer. Order matters: longer, more specific patterns (currency, statutes) must be tried before generic ones (numbers, words), otherwise `₹12,00,000` is consumed by the number rule first.

**Hybrid tokenizer:** spaCy's tokenizer as the base (it handles ordinary English well), followed by merging any span the custom rules recognise. This keeps spaCy's strengths and patches only the domain gaps.

In [37]:
from spacy.util import filter_spans

ABBREVIATIONS = r"(?:Pvt|Ltd|Co|Inc|No|Rs|Sr|Jr|Dr|Mr|Mrs|Ms|viz|approx|Govt|Dept)\.|w\.e\.f\.|i\.e\.|e\.g\.|a\.m\.|p\.m\.|B\.Tech|M\.Tech|B\.E\.|M\.B\.A\.|etc\."

# ORDER MATTERS: most specific first.
CUSTOM_PATTERNS = [
    ("MONEY",   r"(?:₹|Rs\.?|INR)\s?\d[\d,]*(?:\.\d+)?(?:/-)?(?:\s?(?:LPA|lakhs?|crores?))?"),
    ("STATUTE", r"(?:[A-Z][a-z]+\s)+Act,?\s\d{4}|(?:Section|Sec\.)\s\d+[A-Z]?(?:\(\d+\))?"),
    ("CLAUSE",  r"(?:Clause|Article)\s\d+(?:\.\d+)*(?:\([a-z]{1,3}\))?|\b\d+\.\d+(?:\.\d+)*\([a-z]{1,3}\)"),
    ("DATE",    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),
    ("TIME",    r"\b\d{1,2}(?::\d{2})?\s?(?:a\.m\.|p\.m\.)"),
    ("ABBR",    ABBREVIATIONS),
    ("HYPHEN",  r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b"),
    ("NUMBER",  r"\b\d+(?:,\d+)*(?:\.\d+)?\b"),
    ("WORD",    r"[A-Za-z]+(?:'[a-z]+)?"),
    ("PUNCT",   r"[^\w\s]"),
]
CUSTOM_RE = re.compile("|".join(f"(?P<{n}>{p})" for n, p in CUSTOM_PATTERNS))
PROTECTED = {"MONEY", "STATUTE", "CLAUSE", "DATE", "TIME", "ABBR", "HYPHEN"}

def custom_tokenize(text, with_types=False):
    out = [(m.group(), m.lastgroup) for m in CUSTOM_RE.finditer(text)]
    return out if with_types else [t for t, _ in out]

def hybrid_doc(text):
    """spaCy tokenization, then merge every span the custom rules protect."""
    d = nlp.tokenizer(text)
    spans = []
    for m in CUSTOM_RE.finditer(text):
        if m.lastgroup in PROTECTED:
            sp = d.char_span(m.start(), m.end(), alignment_mode="expand")
            if sp is not None and len(sp) > 1:
                spans.append(sp)
    with d.retokenize() as retok:
        for sp in filter_spans(spans):
            retok.merge(sp)
    return d

def hybrid_tokenize(text):
    return [t.text for t in hybrid_doc(text) if not t.is_space]

TOKENIZERS = {
    "nltk":   lambda t: word_tokenize(t),
    "spacy":  lambda t: [tok.text for tok in nlp.tokenizer(t) if not tok.is_space],
    "custom": custom_tokenize,
    "hybrid": hybrid_tokenize,
}
print(custom_tokenize(probe, with_types=True))

[('The', 'WORD'), ('CTC', 'WORD'), ('is', 'WORD'), ('₹12,00,000', 'MONEY'), ('per', 'WORD'), ('annum', 'WORD'), (',', 'PUNCT'), ('subject', 'WORD'), ('to', 'WORD'), ('Section 27', 'STATUTE'), ('and', 'WORD'), ('Clause 4.2(a)', 'CLAUSE'), (';', 'PUNCT'), ('a', 'WORD'), ('non-compete', 'HYPHEN'), ('applies', 'WORD'), ('w.e.f.', 'ABBR'), ('15', 'NUMBER'), ('April', 'WORD'), ('2026', 'NUMBER'), ('.', 'PUNCT')]


### Evaluating the tokenizers against a gold standard

Claiming "custom is better" needs evidence. The cell below scores each tokenizer against hand-written gold tokenizations of domain sentences, using exact-match accuracy and a token-level F1.

> **YOUR INPUT.** `TOKEN_GOLD` holds seed examples drawn from typical employment-contract wording. Add at least five sentences copied from *your own* documents, with the tokenization you consider correct. Your report is stronger if the gold set comes from your corpus.

In [38]:
TOKEN_GOLD = [
    ("The CTC is ₹12,00,000 per annum.",
     ["The", "CTC", "is", "₹12,00,000", "per", "annum", "."]),
    ("Payable as Rs. 8,50,000/- only.",
     ["Payable", "as", "Rs. 8,50,000/-", "only", "."]),
    ("The Employee shall not sign a non-compete.",
     ["The", "Employee", "shall", "not", "sign", "a", "non-compete", "."]),
    ("This is subject to Section 27 of the Indian Contract Act, 1872.",
     ["This", "is", "subject", "to", "Section 27", "of", "the", "Indian Contract Act, 1872", "."]),
    ("Notice is governed by Clause 4.2(a).",
     ["Notice", "is", "governed", "by", "Clause 4.2(a)", "."]),
    ("Acme Pvt. Ltd. appoints the Employee w.e.f. 15 April 2026.",
     ["Acme", "Pvt.", "Ltd.", "appoints", "the", "Employee", "w.e.f.", "15", "April", "2026", "."]),
    ("A B.Tech graduate may work-from-home.",
     ["A", "B.Tech", "graduate", "may", "work-from-home", "."]),
    ("Joining date is 01/04/2026 at 9 a.m.",
     ["Joining", "date", "is", "01/04/2026", "at", "9 a.m.", ]),
]

def token_f1(pred, gold):
    pc, gc = Counter(pred), Counter(gold)
    overlap = sum((pc & gc).values())
    p = overlap / max(len(pred), 1); r = overlap / max(len(gold), 1)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

score_rows, example_rows = [], []
for name, fn in TOKENIZERS.items():
    exact = f1s = 0
    for sent, gold in TOKEN_GOLD:
        pred = fn(sent)
        exact += int(pred == gold)
        f1s += token_f1(pred, gold)
    score_rows.append({"tokenizer": name,
                       "exact_match_%": round(100 * exact / len(TOKEN_GOLD), 1),
                       "token_F1": round(f1s / len(TOKEN_GOLD), 3)})
for sent, gold in TOKEN_GOLD:
    row = {"input": sent, "gold": " | ".join(gold)}
    for name, fn in TOKENIZERS.items():
        row[name] = " | ".join(fn(sent))
    example_rows.append(row)

tok_scores = pd.DataFrame(score_rows)
print("CAUTION: the seed gold set and the custom rules were written together, so custom/hybrid scoring 100%\n"
      "on it is circular, not evidence. The comparison becomes meaningful once you add sentences from YOUR corpus.\n")
tok_examples = pd.DataFrame(example_rows)
save(tok_examples, "tokenization_comparison.csv")
display(tok_scores)
tok_examples

CAUTION: the seed gold set and the custom rules were written together, so custom/hybrid scoring 100%
on it is circular, not evidence. The comparison becomes meaningful once you add sentences from YOUR corpus.

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/tokenization_comparison.csv


,tokenizer,exact_match_%,token_F1
0,nltk,37.5,0.796
1,spacy,0.0,0.714
2,custom,100.0,1.000
3,hybrid,100.0,1.000


,input,gold,nltk,spacy,custom,hybrid
0,"The CTC is ₹12,00,000 per annum.","The | CTC | is | ₹12,00,000 | per | annum | .","The | CTC | is | ₹12,00,000 | per | annum | .","The | CTC | is | ₹ | 12,00,000 | per | annum | .","The | CTC | is | ₹12,00,000 | per | annum | .","The | CTC | is | ₹12,00,000 | per | annum | ."
1,"Payable as Rs. 8,50,000/- only.","Payable | as | Rs. 8,50,000/- | only | .","Payable | as | Rs | . | 8,50,000/- | only | .","Payable | as | Rs | . | 8,50,000/- | only | .","Payable | as | Rs. 8,50,000/- | only | .","Payable | as | Rs. 8,50,000/- | only | ."
2,The Employee shall not sign a non-compete.,The | Employee | shall | not | sign | a | non-compete | .,The | Employee | shall | not | sign | a | non-compete | .,The | Employee | shall | not | sign | a | non | - | compete | .,The | Employee | shall | not | sign | a | non-compete | .,The | Employee | shall | not | sign | a | non-compete | .
3,"This is subject to Section 27 of the Indian Contract Act, 1872.","This | is | subject | to | Section 27 | of | the | Indian Contract Act, 1872 | .","This | is | subject | to | Section | 27 | of | the | Indian | Contract | Act | , | 1872 | .","This | is | subject | to | Section | 27 | of | the | Indian | Contract | Act | , | 1872 | .","This | is | subject | to | Section 27 | of | the | Indian Contract Act, 1872 | .","This | is | subject | to | Section 27 | of | the | Indian Contract Act, 1872 | ."
4,Notice is governed by Clause 4.2(a).,Notice | is | governed | by | Clause 4.2(a) | .,Notice | is | governed | by | Clause | 4.2 | ( | a | ) | .,Notice | is | governed | by | Clause | 4.2(a | ) | .,Notice | is | governed | by | Clause 4.2(a) | .,Notice | is | governed | by | Clause 4.2(a) | .
5,Acme Pvt. Ltd. appoints the Employee w.e.f. 15 April 2026.,Acme | Pvt. | Ltd. | appoints | the | Employee | w.e.f. | 15 | April | 2026 | .,Acme | Pvt | . | Ltd. | appoints | the | Employee | w.e.f | . | 15 | April | 2026 | .,Acme | Pvt | . | Ltd. | appoints | the | Employee | w.e.f | . | 15 | April | 2026 | .,Acme | Pvt. | Ltd. | appoints | the | Employee | w.e.f. | 15 | April | 2026 | .,Acme | Pvt. | Ltd. | appoints | the | Employee | w.e.f. | 15 | April | 2026 | .
6,A B.Tech graduate may work-from-home.,A | B.Tech | graduate | may | work-from-home | .,A | B.Tech | graduate | may | work-from-home | .,A | B.Tech | graduate | may | work | - | from | - | home | .,A | B.Tech | graduate | may | work-from-home | .,A | B.Tech | graduate | may | work-from-home | .
7,Joining date is 01/04/2026 at 9 a.m.,Joining | date | is | 01/04/2026 | at | 9 a.m.,Joining | date | is | 01/04/2026 | at | 9 | a.m | .,Joining | date | is | 01/04/2026 | at | 9 | a.m.,Joining | date | is | 01/04/2026 | at | 9 a.m.,Joining | date | is | 01/04/2026 | at | 9 a.m.


In [39]:
# Corpus-level effect of each tokenizer (same dictionary view as Exercise 1).
rows = []
for name, fn in TOKENIZERS.items():
    toks = [tok for t in docs.values() for tok in fn(t)]
    rows.append({"tokenizer": name, "tokens": len(toks), "token_dictionary": len(set(toks)),
                 "term_dictionary": len(term_dictionary(toks))})
corpus_tok = pd.DataFrame(rows)
display(corpus_tok)

best = tok_scores.sort_values("token_F1", ascending=False).iloc[0]["tokenizer"]
print(f"Highest token F1 on the gold set: {best}")
print("Selected for the rest of the pipeline: HYBRID. Rationale: it keeps spaCy's handling of ordinary English\n"
      "(which the pure-regex custom tokenizer only approximates) while fixing every domain construct in the gold set.\n"
      "If your gold results disagree, change SELECTED_TOKENIZER and justify it in the report.")
SELECTED_TOKENIZER = "hybrid"

,tokenizer,tokens,token_dictionary,term_dictionary
0,nltk,330892,20564,17052
1,spacy,337719,19966,16451
2,custom,343659,18698,15033
3,hybrid,333415,20423,16916


Highest token F1 on the gold set: hybrid
Selected for the rest of the pipeline: HYBRID. Rationale: it keeps spaCy's handling of ordinary English
(which the pure-regex custom tokenizer only approximates) while fixing every domain construct in the gold set.
If your gold results disagree, change SELECTED_TOKENIZER and justify it in the report.


## Exercise 4: BPE tokenization

BPE is trained on this corpus with the HuggingFace `tokenizers` library, and compared with GPT-2's pretrained BPE (trained on general web text). The comparison shows how a general-purpose subword vocabulary fragments legal vocabulary that a corpus-trained vocabulary keeps whole.

`BPE_VOCAB_SIZE` in CONFIG controls the merge budget: a small corpus supports only a small vocabulary before BPE runs out of frequent pairs to merge.

In [40]:
bpe = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe.normalizer = normalizers.NFKC()
bpe.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=CONFIG["BPE_VOCAB_SIZE"], min_frequency=2,
                              special_tokens=["[UNK]"])
bpe.train_from_iterator(list(docs.values()), trainer=trainer)
print(f"Corpus BPE trained. Vocabulary size: {bpe.get_vocab_size():,}")

try:
    gpt2 = Tokenizer.from_pretrained("gpt2")
    print(f"GPT-2 pretrained BPE loaded. Vocabulary size: {gpt2.get_vocab_size():,}")
except Exception as e:
    gpt2 = None
    print(f"GPT-2 tokenizer unavailable ({type(e).__name__}); continuing with corpus BPE only.")

def bpe_tokens(text):  return bpe.encode(text).tokens
def gpt2_tokens(text): return gpt2.encode(text).tokens if gpt2 else []

ALL_TOKENIZERS = dict(TOKENIZERS, bpe=bpe_tokens)
rows = []
for name, fn in ALL_TOKENIZERS.items():
    toks = [tok for t in docs.values() for tok in fn(t)]
    rows.append({"tokenizer": name, "token_count": len(toks), "vocabulary_size": len(set(toks))})
if gpt2:
    toks = [tok for t in docs.values() for tok in gpt2_tokens(t)]
    rows.append({"tokenizer": "gpt2_bpe", "token_count": len(toks), "vocabulary_size": len(set(toks))})
bpe_compare = pd.DataFrame(rows)
bpe_compare

Corpus BPE trained. Vocabulary size: 2,000


tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT-2 pretrained BPE loaded. Vocabulary size: 50,257


,tokenizer,token_count,vocabulary_size
0,nltk,330892,20564
1,spacy,337719,19966
2,custom,343659,18698
3,hybrid,333415,20423
4,bpe,506488,1979
5,gpt2_bpe,417947,15009


In [41]:
# Common, rare and domain-specific words, and how each tokenizer splits them.
word_freq = Counter(t.lower() for t in custom_tokenize(corpus) if t.isalpha() and len(t) > 3)
common = [w for w, _ in word_freq.most_common(8)]
rare = sorted(w for w, c in word_freq.items() if c == 1)[:8]
DOMAIN_WORDS = ["non-compete", "non-solicitation", "confidentiality", "remuneration", "probation",
                "termination", "indemnify", "stipend", "gratuity", "work-from-home", "hereinafter",
                "jurisdiction"]

rows, n = [], 1
for category, words in [("common", common), ("rare", rare), ("domain", DOMAIN_WORDS)]:
    for w in words:
        b = bpe_tokens(w)
        row = {"S.No.": n, "Word": w, "category": category,
               "BPE Tokens": " ".join(b), "bpe_subwords": len(b),
               "nltk": " ".join(word_tokenize(w)), "spacy": " ".join(t.text for t in nlp.tokenizer(w)),
               "custom": " ".join(custom_tokenize(w))}
        if gpt2:
            g = gpt2_tokens(w); row["GPT-2 BPE"] = " ".join(g); row["gpt2_subwords"] = len(g)
        rows.append(row); n += 1
bpe_results = pd.DataFrame(rows)
save(bpe_results, "bpe_results.csv")

agg = bpe_results.groupby("category")[[c for c in ["bpe_subwords", "gpt2_subwords"] if c in bpe_results]].mean().round(2)
print("\nMean subwords per word (lower = the tokenizer 'knows' the word):")
display(agg)
bpe_results

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/bpe_results.csv

Mean subwords per word (lower = the tokenizer 'knows' the word):


,bpe_subwords,gpt2_subwords
category,,
common,1.00,1.12
domain,2.75,3.33
rare,3.75,2.88


,S.No.,Word,category,BPE Tokens,bpe_subwords,nltk,spacy,custom,GPT-2 BPE,gpt2_subwords
0,1,company,common,company,1,company,company,company,company,1
1,2,your,common,your,1,your,your,your,your,1
2,3,this,common,this,1,this,this,this,this,1
3,4,will,common,will,1,will,will,will,will,1
4,5,with,common,with,1,with,with,with,with,1
5,6,shall,common,shall,1,shall,shall,shall,shall,1
6,7,that,common,that,1,that,that,that,that,1
7,8,agreement,common,agreement,1,agreement,agreement,agreement,ag reement,2
8,9,aabcns,rare,a ab c n s,5,aabcns,aabcns,aabcns,a abc ns,3
9,10,aamanagara,rare,a am an agar a,5,aamanagara,aamanagara,aamanagara,a aman agara,3


**What to write up (Exercise 4).** Common words should come out as a single corpus-BPE token, because frequent pairs get merged first. Rare words split into several pieces because they never earned merges. Domain words are the interesting case: they are common *in this corpus* but rare *on the web*, so corpus BPE usually keeps them whole while GPT-2 fragments them. That gap is a direct, measurable argument that general-purpose tokenization loses domain vocabulary. Also note the trade-off: BPE has no unknown words, but its subwords (`termin` + `ation`) are not meaningful units for a Boolean index, which is why this pipeline indexes hybrid tokens and uses BPE for analysis only.

## Exercise 5: When verbs matter, and when only noun phrases do

**(i) Verbs and noun phrases both needed: obligation extraction.** "The Employer **may terminate** the Employee" grants a right; "the Employee **shall not disclose** confidential information" imposes a prohibition. The answer to "who must do what" depends on the subject noun phrase, the modal, the negation, *and* the verb. Dropping verbs would make a permission and a prohibition indistinguishable.

**(ii) Only noun phrases needed: clause-type indexing.** A user searching for the "notice period" or "probation period" wants to find the clause, not an action. Noun phrases like `confidential information`, `intellectual property`, `governing law` identify clause types directly; the verbs around them add nothing to that task.

In [42]:
MODALS = {"shall", "may", "must", "will", "should", "can"}

def obligations(sd):
    rows = []
    for tok in sd:
        if tok.lower_ in MODALS and tok.head.pos_ in {"VERB", "AUX"}:
            verb = tok.head
            subj = next((c for c in verb.children if c.dep_ in {"nsubj", "nsubjpass"}), None)
            neg = any(c.dep_ == "neg" for c in verb.children)
            obj = next((c for c in verb.children if c.dep_ in {"dobj", "attr", "oprd"}), None)
            rows.append({"subject": " ".join(t.text for t in subj.subtree) if subj else "",
                         "modal": tok.text, "negated": neg, "verb": verb.lemma_,
                         "object": " ".join(t.text for t in obj.subtree) if obj else "",
                         "type": ("PROHIBITION" if neg else "OBLIGATION") if tok.lower_ in {"shall", "must"}
                                 else ("RIGHT" if not neg else "RESTRICTED RIGHT")})
    return rows

obl_rows = []
for did, sd in spacy_docs.items():
    for r in obligations(sd):
        obl_rows.append({"doc_id": did, **r})
obl_df = pd.DataFrame(obl_rows)
print(f"(i) Verb + noun phrase task: {len(obl_df)} obligations/rights extracted. Sample:")
display(obl_df.head(12))

np_counts = Counter(" ".join(t.lemma_.lower() for t in chunk if not t.is_stop and not t.is_punct)
                    for sd in spacy_docs.values() for chunk in sd.noun_chunks)
np_counts = Counter({k: v for k, v in np_counts.items() if len(k.split()) >= 2})
print("\n(ii) Noun-phrase-only task: most frequent multi-word noun phrases (clause-type candidates):")
pd.DataFrame(np_counts.most_common(15), columns=["noun_phrase", "frequency"])

(i) Verb + noun phrase task: 6185 obligations/rights extracted. Sample:


,doc_id,subject,modal,negated,verb,object,type
0,D13,You,will,False,be,,RIGHT
1,D13,The office,will,False,close,,RIGHT
2,D13,10 business days ( which includes 5 Government mandated holidays ) in a year and an additional 2 days,can,False,choose,,RIGHT
3,D13,you,can,False,avail,leave for 16 days in your first year as part of the earned leave component,RIGHT
4,D13,Women employees,will,False,entitle,,RIGHT
5,D13,Male employees,will,False,entitle,,RIGHT
6,D13,Your working hours,will,False,be,,RIGHT
7,D13,Your date of joining,will,False,communicate,,RIGHT
8,D13,You,will,False,base,,RIGHT
9,D13,You,will,False,pay,gross emoluments and incentives,RIGHT



(ii) Noun-phrase-only task: most frequent multi-word noun phrases (clause-type candidates):


,noun_phrase,frequency
0,confidential information,298
1,notice period,109
2,sole discretion,103
3,rajarajeswari college,103
4,applicable law,98
5,company policy,89
6,successful completion,76
7,intellectual property,76
8,probation period,75
9,offer letter,70


## Exercise 6: Dates and numbers

**Decision: this domain requires dates and numbers, so they are kept and normalised, not removed.** Salary, notice period, probation length, bond duration, penalty amounts and joining dates are exactly what an employee reviews a contract for. A pipeline that strips digits would make "30 days' notice" and "90 days' notice" identical.

One normalisation is applied: Indian drafting often duplicates numbers in words, as in `6 (six) months`. This is collapsed to `6 months` so the duration is a single consistent pattern.

In [43]:
DUR_PAREN = re.compile(r"\b(\d+)\s*\(\s*[a-z\- ]+\s*\)\s*(days?|weeks?|months?|years?)", re.IGNORECASE)
DURATION = re.compile(r"\b\d+\s+(?:days?|weeks?|months?|years?)\b", re.IGNORECASE)

def normalise_numbers(text):
    return DUR_PAREN.sub(lambda m: f"{m.group(1)} {m.group(2)}", text)

ex = "The Employee shall be on probation for a period of 6 (six) months and give 30 (thirty) days' notice."
print("Before:", ex)
print("After: ", normalise_numbers(ex))

numeric_rows = []
for did, t in docs.items():
    nt = normalise_numbers(t)
    typed = custom_tokenize(nt, with_types=True)
    numeric_rows.append({"doc_id": did,
                         "money": sum(1 for _, k in typed if k == "MONEY"),
                         "dates": sum(1 for _, k in typed if k == "DATE"),
                         "durations": len(DURATION.findall(nt)),
                         "other_numbers": sum(1 for _, k in typed if k == "NUMBER")})
numeric_df = pd.DataFrame(numeric_rows)
print(f"\nDocuments containing at least one money amount: {(numeric_df.money > 0).sum()} of {len(numeric_df)}")
print(f"Documents containing at least one duration:     {(numeric_df.durations > 0).sum()} of {len(numeric_df)}")
numeric_df

Before: The Employee shall be on probation for a period of 6 (six) months and give 30 (thirty) days' notice.
After:  The Employee shall be on probation for a period of 6 months and give 30 days' notice.

Documents containing at least one money amount: 12 of 44
Documents containing at least one duration:     19 of 44


,doc_id,money,dates,durations,other_numbers
0,D01,0,0,0,1
1,D02,0,0,0,2
2,D03,0,0,0,1
3,D04,0,0,0,1
4,D05,0,0,0,3
5,D06,0,0,0,3
6,D07,0,0,0,2
7,D08,0,0,0,1
8,D09,0,0,0,5
9,D10,0,0,0,1


## Exercise 7: Stop-word removal

Stop-word removal is **not** automatically beneficial here. Standard lists were built for general English, where "not" and "without" are low-information. In a contract they carry the legal meaning.

- **NLTK's list** removes `not`, `no`, `nor`, `any`, `all`, `only`, `before`, `after`, `during`, `until`, `against`.
- **spaCy's list** is far more aggressive: it additionally removes `never`, `without`, `unless`, `except`, `none`, `cannot`, `may`, `must`, `within`.

**Custom list used in the final pipeline:** NLTK's list *minus* a protected set (negations, modals, quantifiers, temporal scope words), *plus* archaic drafting words (`hereby`, `herein`, `hereto`, `thereof`, `whereas`) that are frequent in every contract but identify no clause.

The cell below shows the meaning change directly.

In [44]:
from nltk.corpus import stopwords

NLTK_STOP = set(stopwords.words("english"))
SPACY_STOP = set(nlp.Defaults.stop_words)
PROTECT = {"not", "no", "nor", "never", "without", "unless", "except", "neither", "none", "cannot",
           "n't", "shall", "may", "must", "will", "should", "can", "any", "all", "only",
           "before", "after", "during", "within", "until", "against"}
ARCHAIC = {"hereby", "herein", "hereto", "hereof", "thereof", "therein", "whereas", "witnesseth",
           "hereinafter", "aforesaid"}
CUSTOM_STOP = (NLTK_STOP - PROTECT) | ARCHAIC

print(f"NLTK stop list:   {len(NLTK_STOP)} words | protected words it removes: {sorted(PROTECT & NLTK_STOP)}")
print(f"spaCy stop list:  {len(SPACY_STOP)} words | protected words it removes: {sorted(PROTECT & SPACY_STOP)}")
print(f"Custom stop list: {len(CUSTOM_STOP)} words | protected words it removes: {sorted(PROTECT & CUSTOM_STOP)}\n")

MEANING_TESTS = [
    "The Employee shall not disclose any confidential information.",
    "The Employer may terminate the Employee without notice.",
    "No bond amount is payable unless the Employee resigns before 24 months.",
    "The Employee must never solicit any client during the term.",
]
rows = []
for s in MEANING_TESTS:
    toks = [t.lower() for t in word_tokenize(s) if t.isalpha()]
    rows.append({"sentence": s,
                 "NLTK removal": " ".join(t for t in toks if t not in NLTK_STOP),
                 "spaCy removal": " ".join(t for t in toks if t not in SPACY_STOP),
                 "custom removal": " ".join(t for t in toks if t not in CUSTOM_STOP)})
pd.DataFrame(rows)

NLTK stop list:   198 words | protected words it removes: ['after', 'against', 'all', 'any', 'before', 'can', 'during', 'no', 'nor', 'not', 'only', 'should', 'until', 'will']
spaCy stop list:  326 words | protected words it removes: ['after', 'against', 'all', 'any', 'before', 'can', 'cannot', 'during', 'except', 'may', 'must', "n't", 'neither', 'never', 'no', 'none', 'nor', 'not', 'only', 'should', 'unless', 'until', 'will', 'within', 'without']
Custom stop list: 194 words | protected words it removes: []



,sentence,NLTK removal,spaCy removal,custom removal
0,The Employee shall not disclose any confidential information.,employee shall disclose confidential information,employee shall disclose confidential information,employee shall not disclose any confidential information
1,The Employer may terminate the Employee without notice.,employer may terminate employee without notice,employer terminate employee notice,employer may terminate employee without notice
2,No bond amount is payable unless the Employee resigns before 24 months.,bond amount payable unless employee resigns months,bond payable employee resigns months,no bond amount payable unless employee resigns before months
3,The Employee must never solicit any client during the term.,employee must never solicit client term,employee solicit client term,employee must never solicit any client during term


Read the table above aloud. Under **NLTK's** list, sentence 1 becomes "employee shall disclose confidential information": the removal of `not` turns a prohibition into an obligation. Under **spaCy's** list it gets worse: "The Employer may terminate the Employee without notice" becomes "employer terminate employee notice", which a reader would take to mean termination *with* notice. Only the custom list preserves the meaning of every sentence. These examples are the justification for the custom list.

## Exercise 8: Stemming

Three stemmers are compared: **Porter** (conservative, rule-based), **Snowball** (Porter's refinement, "Porter2") and **Lancaster** (aggressive, iterative).

- **Over-stemming:** words with *different* meanings reduced to the *same* stem. The test groups below are pairs that must stay distinct in a contract.
- **Under-stemming:** words with the *same* meaning left with *different* stems, so a search for one misses the other.

In [45]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer
STEMMERS = {"porter": PorterStemmer(), "snowball": SnowballStemmer("english"), "lancaster": LancasterStemmer()}

OVER_GROUPS = [  # must stay DISTINCT
    ["employee", "employer", "employment"], ["confidential", "confidence"], ["terminal", "termination"],
    ["provision", "provide"], ["policy", "police"], ["general", "generate"], ["organ", "organization"],
]
UNDER_GROUPS = [  # should be the SAME
    ["compensate", "compensation"], ["resign", "resignation"], ["solicit", "solicitation"],
    ["indemnify", "indemnity"], ["renew", "renewal"], ["absorb", "absorption"], ["terminate", "termination"],
]

rows = []
for kind, groups in [("over-stemming test", OVER_GROUPS), ("under-stemming test", UNDER_GROUPS)]:
    for g in groups:
        row = {"test": kind, "words": ", ".join(g)}
        for name, st in STEMMERS.items():
            stems = [st.stem(w) for w in g]
            row[name] = ", ".join(stems)
            collapsed = len(set(stems)) < len(g)
            row[f"{name}_error"] = collapsed if kind.startswith("over") else not collapsed
        rows.append(row)
stem_tests = pd.DataFrame(rows)

err = {name: {"over-stemming errors": int(stem_tests.loc[stem_tests.test.str.startswith("over"), f"{name}_error"].sum()),
              "under-stemming errors": int(stem_tests.loc[stem_tests.test.str.startswith("under"), f"{name}_error"].sum())}
       for name in STEMMERS}
display(pd.DataFrame(err).T)
stem_tests[["test", "words"] + list(STEMMERS)]

,over-stemming errors,under-stemming errors
porter,4,2
snowball,3,2
lancaster,6,0


,test,words,porter,snowball,lancaster
0,over-stemming test,"employee, employer, employment","employe, employ, employ","employe, employ, employ","employ, employ, employ"
1,over-stemming test,"confidential, confidence","confidenti, confid","confidenti, confid","confid, confid"
2,over-stemming test,"terminal, termination","termin, termin","termin, termin","termin, termin"
3,over-stemming test,"provision, provide","provis, provid","provis, provid","provid, provid"
4,over-stemming test,"policy, police","polici, polic","polici, polic","policy, pol"
5,over-stemming test,"general, generate","gener, gener","general, generat","gen, gen"
6,over-stemming test,"organ, organization","organ, organ","organ, organ","org, org"
7,under-stemming test,"compensate, compensation","compens, compens","compens, compens","compens, compens"
8,under-stemming test,"resign, resignation","resign, resign","resign, resign","resign, resign"
9,under-stemming test,"solicit, solicitation","solicit, solicit","solicit, solicit","solicit, solicit"


**Selection rule.** Pick the stemmer with the fewest *over-stemming* errors (ties broken by fewer under-stemming errors), because in this domain over-stemming is the costlier failure: conflating the Employer with the concept of employment, or a confidentiality obligation with a statement of confidence, silently merges distinct legal meanings. Under-stemming only costs recall, and lemmatization (Exercise 10) addresses it more safely. The cell below applies this rule, and the selected stemmer is the one Pipeline A uses. Note that *no* stemmer keeps `employer` and `employment` apart, which is the main argument for lemmatization.

In [46]:
err_df = pd.DataFrame(err).T
SELECTED_STEMMER = err_df.sort_values(["over-stemming errors", "under-stemming errors"]).index[0]
stemmer = STEMMERS[SELECTED_STEMMER]
print(f"Selected stemmer for Pipeline A: {SELECTED_STEMMER} "
      f"({err_df.loc[SELECTED_STEMMER, 'over-stemming errors']} over-stemming, "
      f"{err_df.loc[SELECTED_STEMMER, 'under-stemming errors']} under-stemming errors)")

Selected stemmer for Pipeline A: snowball (3 over-stemming, 2 under-stemming errors)


## Exercise 9: Order of stop-word removal and stemming

**Stop-word removal must come before stemming.** Stop lists contain whole words. Stemming first rewrites them into forms the list no longer recognises: Porter turns `was` into `wa`, `this` into `thi`, `has` into `ha`, `does` into `doe`. These then survive removal and pollute the vocabulary. The experiment below measures it using Porter, whose rewrites make the effect easiest to see; the conclusion about order holds for any stemmer.

In [47]:
porter = STEMMERS["porter"]
base = [t.lower() for t in word_tokenize(corpus) if t.isalpha()]

stop_then_stem = [porter.stem(t) for t in base if t not in NLTK_STOP]
stem_then_stop = [s for s in (porter.stem(t) for t in base) if s not in NLTK_STOP]
leaked = Counter(porter.stem(t) for t in base if t in NLTK_STOP and porter.stem(t) not in NLTK_STOP)

order_df = pd.DataFrame([
    {"order": "stop-word removal -> stemming", "tokens": len(stop_then_stem), "vocabulary": len(set(stop_then_stem)),
     "stop words surviving": 0},
    {"order": "stemming -> stop-word removal", "tokens": len(stem_then_stop), "vocabulary": len(set(stem_then_stop)),
     "stop words surviving": sum(leaked.values())},
])
display(order_df)
print("Stop words that survived because stemming ran first (stem: count):")
print(dict(leaked.most_common(12)))

,order,tokens,vocabulary,stop words surviving
0,stop-word removal -> stemming,155041,9054,0
1,stemming -> stop-word removal,163117,9038,8293


Stop words that survived because stemming ran first (stem: count):
{'ani': 3531, 'thi': 2544, 'dure': 656, 'abov': 388, 'ha': 335, 'onli': 312, 'hi': 152, 'befor': 150, 'doe': 70, 'wa': 67, 'onc': 53, 'veri': 21}


## Exercise 10: Lemmatization

Three lemmatizers are compared: **WordNet without POS** (treats every word as a noun), **WordNet with POS** (POS from NLTK's tagger) and **spaCy** (POS-aware rule and lookup lemmatizer). Lemmatization returns dictionary forms, so unlike stemming it cannot produce non-words like `employe`, and it keeps `employer` and `employment` apart.

In [48]:
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet as wn

wnl = WordNetLemmatizer()
def wn_pos(tag):
    return {"J": wn.ADJ, "V": wn.VERB, "N": wn.NOUN, "R": wn.ADV}.get(tag[0], wn.NOUN)

TABLE_C_WORDS = ["employees", "employer", "employment", "terminated", "terminating", "agreed", "obligations",
                 "confidential", "provisions", "resigned", "compensation", "was", "binding", "better",
                 "solicited", "renewed", "indemnities", "paid"]
rows = []
for w in TABLE_C_WORDS:
    tag = pos_tag([w])[0][1]
    rows.append({"Word": w,
                 "Porter": STEMMERS["porter"].stem(w), "Snowball": STEMMERS["snowball"].stem(w),
                 "Lancaster": STEMMERS["lancaster"].stem(w),
                 "WordNet (no POS)": wnl.lemmatize(w),
                 "WordNet (POS)": wnl.lemmatize(w, wn_pos(tag)),
                 "spaCy": nlp(w)[0].lemma_})
table_c = pd.DataFrame(rows)
save(table_c, "stemming_lemmatization.csv")
table_c

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/stemming_lemmatization.csv


,Word,Porter,Snowball,Lancaster,WordNet (no POS),WordNet (POS),spaCy
0,employees,employe,employe,employ,employee,employee,employee
1,employer,employ,employ,employ,employer,employer,employer
2,employment,employ,employ,employ,employment,employment,employment
3,terminated,termin,termin,termin,terminated,terminate,terminate
4,terminating,termin,termin,termin,terminating,terminate,terminate
5,agreed,agre,agre,agree,agreed,agree,agree
6,obligations,oblig,oblig,oblig,obligation,obligation,obligation
7,confidential,confidenti,confidenti,confid,confidential,confidential,confidential
8,provisions,provis,provis,provid,provision,provision,provision
9,resigned,resign,resign,resign,resigned,resign,resign


**Selection: spaCy lemmatization.** WordNet without POS leaves verbs unchanged (`terminated` stays `terminated`) because it assumes a noun. WordNet with POS fixes that, but here the tag comes from tagging a single word out of context, which is unreliable. spaCy lemmatizes each word *in its sentence*, which is how the pipeline actually uses it. Isolated-word tables like this one understate spaCy's advantage, since its tagger needs context.

### The three pipelines

Defined here because Exercise 11 needs the final pipeline, and Module 3 compares all three. The **order of every step is deliberate**; Module 3 justifies each placement.

| Step | Pipeline A (baseline) | Pipeline B (assignment branch 2) | Final pipeline |
|---|---|---|---|
| Cleaning | yes | yes | yes |
| Number normalisation | no | no | yes (`6 (six) months` to `6 months`) |
| Tokenization | NLTK `word_tokenize` | spaCy | **Hybrid** (spaCy + custom rules) |
| Lemmatize / stem | Stemming (selected in Ex. 8) | spaCy lemmatization | spaCy lemmatization |
| Case | lowercase | lowercase | lowercase, *after* lemmatization |
| Stop words | NLTK list | **none removed** | **custom list** (negations and modals kept) |

In [49]:
def run_pipes(d):
    """Run the loaded spaCy components over an already-tokenized Doc."""
    for _, proc in nlp.pipeline:
        d = proc(d)
    return d

def pipeline_A_sents(text):
    out = []
    for sent in sent_tokenize(clean_text(text)):
        toks = [t.lower() for t in word_tokenize(sent) if re.search(r"\w", t)]
        out.append([stemmer.stem(t) for t in toks if t not in NLTK_STOP])
    return [x for x in out if x]

def pipeline_B_sents(text):
    d = nlp(clean_text(text))
    out = [[t.lemma_.lower() for t in sent if not t.is_space and not t.is_punct] for sent in d.sents]
    return [x for x in out if x]

def final_doc(text):
    return run_pipes(hybrid_doc(normalise_numbers(clean_text(text))))

def final_token(t):
    # merged domain tokens (money, statutes, hyphenated terms) keep their surface form
    return t.text.lower() if (" " in t.text or "-" in t.text or not t.is_alpha) else t.lemma_.lower()

def pipeline_final_sents(text):
    out = []
    for sent in final_doc(text).sents:
        toks = [final_token(t) for t in sent if not t.is_space and not t.is_punct]
        out.append([t for t in toks if t not in CUSTOM_STOP])
    return [x for x in out if x]

# Flat token streams (for counting and indexing) are the sentences concatenated.
def flatten(sents): return [t for s in sents for t in s]
def pipeline_A(text):     return flatten(pipeline_A_sents(text))
def pipeline_B(text):     return flatten(pipeline_B_sents(text))
def pipeline_final(text): return flatten(pipeline_final_sents(text))

PIPELINES = {"Pipeline A": pipeline_A, "Pipeline B": pipeline_B, "Final Pipeline": pipeline_final}
PIPELINE_SENTS = {"Pipeline A": pipeline_A_sents, "Pipeline B": pipeline_B_sents, "Final Pipeline": pipeline_final_sents}
processed, processed_sents, pipe_time = {}, {}, {}
for name, fn in PIPELINE_SENTS.items():
    t0 = time.perf_counter()
    processed_sents[name] = {did: fn(t) for did, t in docs.items()}
    processed[name] = {did: flatten(ss) for did, ss in processed_sents[name].items()}
    pipe_time[name] = time.perf_counter() - t0
    print(f"{name:15s} processed {len(docs)} documents in {pipe_time[name]:.2f}s")

s = MEANING_TESTS[1]
print(f"\n'{s}'")
for name, fn in PIPELINES.items():
    print(f"  {name:15s} {fn(s)}")

Pipeline A      processed 44 documents in 3.31s
Pipeline B      processed 44 documents in 58.99s
Final Pipeline  processed 44 documents in 53.79s

'The Employer may terminate the Employee without notice.'
  Pipeline A      ['employ', 'may', 'termin', 'employe', 'without', 'notic']
  Pipeline B      ['the', 'employer', 'may', 'terminate', 'the', 'employee', 'without', 'notice']
  Final Pipeline  ['employer', 'may', 'terminate', 'employee', 'without', 'notice']


## Exercise 11: Corpus before and after preprocessing (Table A)

"Before" is spaCy tokenization of the cleaned text (from Module 1). "After" is the final pipeline. Unique tokens counts distinct surface strings; vocabulary size counts distinct *terms* (lowercased, punctuation removed), so the two differ most in the "before" column.

In [50]:
before_tokens = all_tokens
after_tokens = [t for toks in processed["Final Pipeline"].values() for t in toks]
table_a = pd.DataFrame([
    {"Measure": "Documents", "Before Processing": len(docs), "After Processing": len(processed["Final Pipeline"])},
    {"Measure": "Tokens", "Before Processing": len(before_tokens), "After Processing": len(after_tokens)},
    {"Measure": "Unique Tokens", "Before Processing": len(set(before_tokens)), "After Processing": len(set(after_tokens))},
    {"Measure": "Vocabulary Size", "Before Processing": len(term_dictionary(before_tokens)),
     "After Processing": len(term_dictionary(after_tokens))},
])
table_a["Change %"] = (100 * (table_a["After Processing"] - table_a["Before Processing"])
                       / table_a["Before Processing"]).round(1)
save(table_a, "preprocessing_results.csv")
table_a

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/preprocessing_results.csv


,Measure,Before Processing,After Processing,Change %
0,Documents,44,44,0.0
1,Tokens,337719,178427,-47.2
2,Unique Tokens,19966,15407,-22.8
3,Vocabulary Size,16451,15375,-6.5


## Exercise 12: POS tagging and custom POS taggers

All taggers here use the **Penn Treebank tagset** (NN, VB, MD, ...), because it is shared by NLTK's tagger, spaCy's fine-grained `tag_`, and the NLTK Treebank corpus used to train the ML tagger. One tagset means one gold standard and directly comparable accuracies.

**Domain terms likely to be mis-tagged:**

| Term | Problem | Correct in context |
|---|---|---|
| `notice`, `bond` | Noun/verb ambiguity | NN after a determiner, VB after a modal |
| `Employee`, `Employer`, `Intern` | Capitalised defined terms get tagged NNP | NN (see note below) |
| `CTC` | Unknown acronym | NN |
| `non-compete`, `non-solicitation` | Hyphenated, often tagged JJ | NN when it names the agreement or clause |
| `w.e.f.` | Unknown abbreviation | IN (it means "with effect from") |
| `₹12,00,000` | Currency token | CD |

> **Annotation choice (state this in your report).** Defined party terms such as "the Employee" are tagged **NN** in the gold standard, not NNP. They are capitalised by drafting convention but behave as role nouns: they take a determiner and refer to whoever fills the role. This is a defensible choice, not the only one. If you prefer NNP, change the gold consistently.

> **YOUR INPUT.** `POS_GOLD_TEST` is the evaluation set and `POS_DOMAIN_TRAIN` is extra training data for the ML tagger. Both are seed examples. Verify every tag, and add sentences from your own documents. Keep the two sets disjoint, or the ML accuracy is meaningless.

In [51]:
POS_GOLD_TEST = [
    [("The","DT"),("Employee","NN"),("shall","MD"),("not","RB"),("sign","VB"),("a","DT"),("non-compete","NN"),(".",".")],
    [("The","DT"),("Company","NN"),("will","MD"),("pay","VB"),("a","DT"),("CTC","NN"),("of","IN"),("₹12,00,000","CD"),("per","IN"),("annum","NN"),(".",".")],
    [("Either","DT"),("party","NN"),("may","MD"),("terminate","VB"),("this","DT"),("Agreement","NN"),("by","IN"),("giving","VBG"),("notice","NN"),(".",".")],
    [("The","DT"),("Employee","NN"),("shall","MD"),("serve","VB"),("the","DT"),("bond","NN"),("for","IN"),("24","CD"),("months","NNS"),(".",".")],
    [("The","DT"),("Intern","NN"),("will","MD"),("receive","VB"),("a","DT"),("stipend","NN"),("w.e.f.","IN"),("15","CD"),("April","NNP"),("2026","CD"),(".",".")],
    [("The","DT"),("Employee","NN"),("shall","MD"),("not","RB"),("solicit","VB"),("any","DT"),("client","NN"),(".",".")],
    [("Confidential","JJ"),("information","NN"),("shall","MD"),("not","RB"),("be","VB"),("disclosed","VBN"),(".",".")],
    [("The","DT"),("Employer","NN"),("shall","MD"),("notice","VB"),("the","DT"),("breach","NN"),("promptly","RB"),(".",".")],
    [("A","DT"),("non-solicitation","NN"),("clause","NN"),("applies","VBZ"),("during","IN"),("the","DT"),("notice","NN"),("period","NN"),(".",".")],
    [("The","DT"),("Consultant","NN"),("shall","MD"),("bond","VB"),("the","DT"),("materials","NNS"),(".",".")],
]
POS_DOMAIN_TRAIN = [
    [("The","DT"),("Freelancer","NN"),("shall","MD"),("not","RB"),("disclose","VB"),("any","DT"),("data","NNS"),(".",".")],
    [("The","DT"),("Employer","NN"),("may","MD"),("notice","VB"),("a","DT"),("delay","NN"),(".",".")],
    [("The","DT"),("Worker","NN"),("must","MD"),("sign","VB"),("the","DT"),("bond","NN"),(".",".")],
    [("A","DT"),("CTC","NN"),("of","IN"),("₹8,00,000","CD"),("is","VBZ"),("payable","JJ"),(".",".")],
    [("The","DT"),("non-compete","NN"),("lasts","VBZ"),("12","CD"),("months","NNS"),("w.e.f.","IN"),("today","NN"),(".",".")],
    [("The","DT"),("Contractor","NN"),("shall","MD"),("bond","VB"),("the","DT"),("equipment","NN"),(".",".")],
]

def tag_spacy(words):
    return [t.tag_ for t in run_pipes(spacy.tokens.Doc(nlp.vocab, words=words))]

def tag_nltk(words):
    return [t for _, t in pos_tag(words)]

# ---- Custom tagger 1: rule/dictionary-based, layered on top of spaCy ----
DOMAIN_LEXICON = {w: "NN" for w in ["employee", "employer", "company", "intern", "consultant", "contractor",
                                     "freelancer", "worker", "candidate", "agreement", "ctc", "stipend",
                                     "non-compete", "non-solicitation", "gratuity", "annum"]}
DOMAIN_LEXICON.update({"w.e.f.": "IN"})
NOUN_VERB = {"notice", "bond", "sign", "release", "claim", "breach", "service"}
MONEY_TOKEN = re.compile(r"^(?:₹|Rs\.?|INR)\s?\d")

def tag_rules(words):
    tags = tag_spacy(words)
    for i, w in enumerate(words):
        lw = w.lower()
        if MONEY_TOKEN.match(w):
            tags[i] = "CD"
        elif lw in DOMAIN_LEXICON:
            tags[i] = DOMAIN_LEXICON[lw]
        elif lw in NOUN_VERB and i > 0:
            prev = tags[i - 1]
            if prev in {"MD", "TO"} or (prev == "RB" and i > 1 and tags[i - 2] == "MD"):
                tags[i] = "VB"
            elif prev in {"DT", "JJ", "PRP$", "POS"}:
                tags[i] = "NN"
    return tags

print("spaCy :", tag_spacy(["The", "Employee", "shall", "bond", "the", "CTC", "."]))
print("rules :", tag_rules(["The", "Employee", "shall", "bond", "the", "CTC", "."]))

spaCy : ['DT', 'NN', 'MD', 'VB', 'DT', 'NNP', '.']
rules : ['DT', 'NN', 'MD', 'VB', 'DT', 'NN', '.']


In [52]:
# ---- Custom tagger 2: ML (multinomial logistic regression on hand-crafted features) ----
from nltk.corpus import treebank

def features(words, i):
    w = words[i]
    return {"w": w.lower(), "suf3": w[-3:].lower(), "suf2": w[-2:].lower(), "pre2": w[:2].lower(),
            "is_title": w.istitle(), "is_upper": w.isupper(), "has_digit": any(c.isdigit() for c in w),
            "has_hyphen": "-" in w, "has_currency": bool(MONEY_TOKEN.match(w)),
            "in_domain_lexicon": w.lower() in DOMAIN_LEXICON,
            "prev": words[i - 1].lower() if i > 0 else "<S>",
            "prev2": words[i - 2].lower() if i > 1 else "<S>",
            "next": words[i + 1].lower() if i < len(words) - 1 else "</S>"}

def to_xy(tagged_sents):
    X, y = [], []
    for sent in tagged_sents:
        sent = [(w, t) for w, t in sent if t != "-NONE-"]      # drop Treebank trace tokens
        words = [w for w, _ in sent]
        for i, (_, t) in enumerate(sent):
            X.append(features(words, i)); y.append(t)
    return X, y

tb = treebank.tagged_sents()
t0 = time.perf_counter()
X_tb, y_tb = to_xy(tb)
vec_general = DictVectorizer()
ml_general = LogisticRegression(max_iter=300).fit(vec_general.fit_transform(X_tb), y_tb)

# Domain-adapted: same features, Treebank plus domain sentences (upweighted so a handful counts)
X_dom, y_dom = to_xy(POS_DOMAIN_TRAIN)
DOMAIN_WEIGHT = 20
vec_domain = DictVectorizer()
X_all = X_tb + X_dom; y_all = y_tb + y_dom
w_all = [1.0] * len(X_tb) + [float(DOMAIN_WEIGHT)] * len(X_dom)
ml_domain = LogisticRegression(max_iter=300).fit(vec_domain.fit_transform(X_all), y_all, sample_weight=w_all)
print(f"Trained on {len(X_tb):,} Treebank tokens (+{len(X_dom)} domain tokens) in {time.perf_counter()-t0:.1f}s")

def tag_ml(model, vec):
    return lambda words: list(model.predict(vec.transform([features(words, i) for i in range(len(words))])))

TAGGERS = {"NLTK default": tag_nltk, "spaCy default": tag_spacy, "Custom rules": tag_rules,
           "Custom ML (Treebank)": tag_ml(ml_general, vec_general),
           "Custom ML (Treebank + domain)": tag_ml(ml_domain, vec_domain)}

Trained on 94,084 Treebank tokens (+44 domain tokens) in 102.2s


In [53]:
DOMAIN_TERMS = set(DOMAIN_LEXICON) | NOUN_VERB | {"shall"}
acc_rows, detail_rows = [], []
for name, fn in TAGGERS.items():
    correct = total = dom_correct = dom_total = 0
    for sent in POS_GOLD_TEST:
        words = [w for w, _ in sent]; gold = [t for _, t in sent]
        pred = fn(words)
        for w, g, p in zip(words, gold, pred):
            total += 1; correct += (g == p)
            if w.lower() in DOMAIN_TERMS or MONEY_TOKEN.match(w):
                dom_total += 1; dom_correct += (g == p)
    acc_rows.append({"tagger": name, "accuracy_%": round(100 * correct / total, 1),
                     "domain_term_accuracy_%": round(100 * dom_correct / max(dom_total, 1), 1)})
pos_acc = pd.DataFrame(acc_rows)
display(pos_acc)

# Table D: every domain-term token in the gold set, default vs custom
for sent in POS_GOLD_TEST:
    words = [w for w, _ in sent]; gold = [t for _, t in sent]
    d, r, m = tag_spacy(words), tag_rules(words), TAGGERS["Custom ML (Treebank + domain)"](words)
    for i, w in enumerate(words):
        if w.lower() in DOMAIN_TERMS or MONEY_TOKEN.match(w):
            detail_rows.append({"Word": w, "Context": " ".join(words), "Gold": gold[i],
                                "Default POS (spaCy)": d[i], "Custom POS (rules)": r[i], "Custom POS (ML)": m[i],
                                "Default Correct": d[i] == gold[i], "Rules Correct": r[i] == gold[i],
                                "ML Correct": m[i] == gold[i]})
table_d = pd.DataFrame(detail_rows)
save(table_d, "pos_tagging_results.csv")
base = pos_acc.set_index("tagger")["accuracy_%"]
print(f"\nImprovement of rule tagger over spaCy default: {base['Custom rules'] - base['spaCy default']:+.1f} points")
table_d

,tagger,accuracy_%,domain_term_accuracy_%
0,NLTK default,82.0,50.0
1,spaCy default,89.9,67.9
2,Custom rules,100.0,100.0
3,Custom ML (Treebank),88.8,67.9
4,Custom ML (Treebank + domain),97.8,96.4


  saved -> /content/drive/MyDrive/NLP_Assessment1/results/pos_tagging_results.csv

Improvement of rule tagger over spaCy default: +10.1 points


,Word,Context,Gold,Default POS (spaCy),Custom POS (rules),Custom POS (ML),Default Correct,Rules Correct,ML Correct
0,Employee,The Employee shall not sign a non-compete .,NN,NN,NN,NN,True,True,True
1,shall,The Employee shall not sign a non-compete .,MD,MD,MD,MD,True,True,True
2,sign,The Employee shall not sign a non-compete .,VB,VB,VB,VB,True,True,True
3,non-compete,The Employee shall not sign a non-compete .,NN,NN,NN,NN,True,True,True
4,Company,"The Company will pay a CTC of ₹12,00,000 per annum .",NN,NNP,NN,NN,False,True,True
5,CTC,"The Company will pay a CTC of ₹12,00,000 per annum .",NN,NNP,NN,NN,False,True,True
6,"₹12,00,000","The Company will pay a CTC of ₹12,00,000 per annum .",CD,NNP,CD,CD,False,True,True
7,annum,"The Company will pay a CTC of ₹12,00,000 per annum .",NN,FW,NN,NN,False,True,True
8,Agreement,Either party may terminate this Agreement by giving notice .,NN,NNP,NN,NN,False,True,True
9,notice,Either party may terminate this Agreement by giving notice .,NN,NN,NN,NN,True,True,True


**Read the rule tagger's score with care.** The dictionary rules were written with the seed gold sentences in view, so a near-perfect score on *this* set is partly circular. Its real test is sentences from your own documents that you tag *before* looking at its output. The ML tagger's number is more honest, because it never saw the test sentences.

**What to write up.** Report overall *and* domain-term accuracy: overall accuracy is dominated by easy tokens (`the`, `.`) that every tagger gets right, so domain-term accuracy is where the custom taggers show their value. Also compare the two ML variants: the gap between "Treebank" and "Treebank + domain" measures how much a handful of in-domain sentences helps, which is the ML version of the argument for domain adaptation. A small gold set means these percentages move a lot per token; say so.

## Exercise 13: Named Entity Recognition

spaCy's default NER is compared with a **custom pipeline**: the same model plus a rule component that adds four domain entities and overrides default predictions where they overlap.

| Label | Source | Example |
|---|---|---|
| PERSON, ORG, GPE, DATE, PRODUCT, EVENT | spaCy default | Rahul Sharma, Acme Technologies Pvt. Ltd., Bengaluru |
| **MONEY** (overridden) | custom regex | `₹12,00,000`, `Rs. 8,50,000/-`, `INR 15 LPA` |
| **DATE** (supplemented) | custom regex | `01/04/2026` (spaCy misses slash-format dates) |
| **DURATION** (new) | custom regex | `30 days`, `6 (six) months` |
| **LAW** | custom regex | `Section 27`, `Indian Contract Act, 1872` |
| **DESIGNATION** (new) | custom regex | `Senior Software Engineer` |

DURATION is deliberately separate from DATE. "30 days' notice" is a length of time, not a point in time, and conflating them would make a query for joining dates return notice periods.

In [54]:
nlp_ner = spacy.load("en_core_web_sm")
DOMAIN_ENT_PATTERNS = [
    ("MONEY", CUSTOM_PATTERNS[0][1]),
    ("DATE", r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),       # spaCy misses slash dates common in Indian contracts
    ("LAW", r"(?:Section|Sec\.)\s\d+[A-Z]?(?:\(\d+\))?|(?:[A-Z][a-z]+\s)+Act,?\s\d{4}"),
    ("DURATION", r"\b\d+\s*(?:\([a-z\- ]+\)\s*)?(?:days?|weeks?|months?|years?)\b"),
    ("DESIGNATION", r"\b(?:(?:Senior|Junior|Lead|Principal|Associate|Chief)\s)?(?:(?:Software|Data|Research|Sales|"
                    r"Marketing|HR|Finance|Product|Technical)\s)?(?:Engineer|Developer|Analyst|Manager|Consultant|"
                    r"Executive|Scientist|Designer|Officer|Architect)\b"),
]

@Language.component("domain_entities")
def domain_entities(doc):
    new = []
    for label, pat in DOMAIN_ENT_PATTERNS:
        for m in re.finditer(pat, doc.text):
            sp = doc.char_span(m.start(), m.end(), label=label, alignment_mode="expand")
            if sp is not None:
                new.append(sp)
    kept = [e for e in doc.ents if not any(e.start < n.end and n.start < e.end for n in new)]
    doc.ents = filter_spans(new + kept)
    return doc

nlp_custom = spacy.load("en_core_web_sm")
nlp_custom.add_pipe("domain_entities", after="ner")
print("Custom NER pipeline:", nlp_custom.pipe_names)

s = "Priya Nair shall pay Rs. 2,00,000 if she leaves before 24 months, subject to Section 27."
print("\nDefault:", [(e.text, e.label_) for e in nlp_ner(s).ents])
print("Custom: ", [(e.text, e.label_) for e in nlp_custom(s).ents])

Custom NER pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner', 'domain_entities']

Default: [('Priya Nair', 'PERSON'), ('2,00,000', 'CARDINAL'), ('24 months', 'DATE'), ('Section 27', 'LAW')]
Custom:  [('Priya Nair', 'PERSON'), ('Rs. 2,00,000', 'MONEY'), ('24 months', 'DURATION'), ('Section 27', 'LAW')]


> **YOUR INPUT.** `NER_GOLD` below is a seed gold standard. Add sentences from your own documents with every entity you consider correct. The scores are only as credible as this set.

In [55]:
NER_GOLD = [
    ("This Agreement is made on 01/04/2026 between Acme Technologies Pvt. Ltd. and Rahul Sharma.",
     [("01/04/2026", "DATE"), ("Acme Technologies Pvt. Ltd.", "ORG"), ("Rahul Sharma", "PERSON")]),
    ("The Employee shall receive a CTC of ₹12,00,000 per annum.", [("₹12,00,000", "MONEY")]),
    ("The Employee is appointed as Senior Software Engineer in Bengaluru.",
     [("Senior Software Engineer", "DESIGNATION"), ("Bengaluru", "GPE")]),
    ("Either party may terminate this Agreement by giving 30 days notice.", [("30 days", "DURATION")]),
    ("This is subject to Section 27 of the Indian Contract Act, 1872.",
     [("Section 27", "LAW"), ("Indian Contract Act, 1872", "LAW")]),
    ("Priya Nair shall pay Rs. 2,00,000 towards training costs if she leaves before 24 months.",
     [("Priya Nair", "PERSON"), ("Rs. 2,00,000", "MONEY"), ("24 months", "DURATION")]),
    ("The courts at Mumbai shall have exclusive jurisdiction.", [("Mumbai", "GPE")]),
    ("The Intern will receive a stipend of ₹15,000 per month from Zenith Software Private Limited.",
     [("₹15,000", "MONEY"), ("Zenith Software Private Limited", "ORG")]),
]

def score_ner(model):
    tot = cor = inc = mis = 0
    for sent, gold in NER_GOLD:
        pred = {(e.text, e.label_) for e in model(sent).ents}
        g = set(gold)
        cor += len(pred & g); inc += len(pred - g); mis += len(g - pred); tot += len(g)
    p = cor / max(cor + inc, 1); r = cor / max(tot, 1)
    return {"gold entities": tot, "correct": cor, "incorrect (spurious or wrong type/span)": inc,
            "missed": mis, "precision": round(p, 3), "recall": round(r, 3),
            "F1": round(0 if p + r == 0 else 2 * p * r / (p + r), 3)}

ner_scores = pd.DataFrame({"spaCy default": score_ner(nlp_ner), "Custom": score_ner(nlp_custom)}).T
display(ner_scores)

rows = []
for sent, gold in NER_GOLD:
    d = {e.text: e.label_ for e in nlp_ner(sent).ents}
    c = {e.text: e.label_ for e in nlp_custom(sent).ents}
    for text, label in gold:
        rows.append({"Entity": text, "Gold Type": label, "Default Predicted": d.get(text, "(missed)"),
                     "Custom Predicted": c.get(text, "(missed)"),
                     "Default Correct?": d.get(text) == label, "Custom Correct?": c.get(text) == label})
table_e_gold = pd.DataFrame(rows)
table_e_gold

,gold entities,correct,incorrect (spurious or wrong type/span),missed,precision,recall,F1
spaCy default,15.0,7.0,11.0,8.0,0.389,0.467,0.424
Custom,15.0,15.0,4.0,0.0,0.789,1.000,0.882


,Entity,Gold Type,Default Predicted,Custom Predicted,Default Correct?,Custom Correct?
0,01/04/2026,DATE,(missed),DATE,False,True
1,Acme Technologies Pvt. Ltd.,ORG,ORG,ORG,True,True
2,Rahul Sharma,PERSON,PERSON,PERSON,True,True
3,"₹12,00,000",MONEY,(missed),MONEY,False,True
4,Senior Software Engineer,DESIGNATION,(missed),DESIGNATION,False,True
5,Bengaluru,GPE,GPE,GPE,True,True
6,30 days,DURATION,DATE,DURATION,False,True
7,Section 27,LAW,LAW,LAW,True,True
8,"Indian Contract Act, 1872",LAW,(missed),LAW,False,True
9,Priya Nair,PERSON,PERSON,PERSON,True,True


In [56]:
# Corpus-wide entity inventory. "Correct?" is left blank for you to fill in by hand (YOUR INPUT).
inv = Counter()
for did, t in docs.items():
    for e in nlp_custom(t).ents:
        inv[(e.text, e.label_)] += 1
default_types = {}
for t in docs.values():
    for e in nlp_ner(t).ents:
        default_types.setdefault(e.text, e.label_)
table_e = pd.DataFrame([{"Entity": txt, "Predicted Type (custom)": lab,
                         "Predicted Type (default)": default_types.get(txt, "(missed)"),
                         "Frequency": n, "Correct?": ""} for (txt, lab), n in inv.most_common()])
save(table_e, "ner_results.csv")
print("Entity counts by type (custom pipeline):")
display(table_e.groupby("Predicted Type (custom)")["Frequency"].sum().sort_values(ascending=False))
table_e.head(25)

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/ner_results.csv
Entity counts by type (custom pipeline):


,Frequency
Predicted Type (custom),
ORG,9113
CARDINAL,2514
DATE,2156
PERSON,2118
GPE,1443
DESIGNATION,829
MONEY,592
WORK_OF_ART,523
DURATION,464


,Entity,Predicted Type (custom),Predicted Type (default),Frequency,Correct?
0,Company,ORG,ORG,1206,
1,Employee,GPE,GPE,421,
2,Executive,DESIGNATION,ORG,344,
3,Consultant,DESIGNATION,PRODUCT,293,
4,India,GPE,GPE,263,
5,TCS,ORG,ORG,225,
6,Agreement,PRODUCT,PRODUCT,215,
7,MOU,ORG,ORG,202,
8,Party,ORG,ORG,172,
9,Confidential Information,ORG,ORG,149,


## Exercise 14: N-gram analysis

N-grams are generated from the **final pipeline's** tokens, per *sentence*, so no n-gram spans two sentences or two documents. (Without this, a clause ending and the next clause heading fuse into junk such as `april 2026 compensation`.) Because the final pipeline keeps negations and modals, phrases like `shall not disclose` survive; under Pipeline A they would collapse to `disclos`.

A **meaningful** n-gram is defined here as one that neither starts nor ends with a function word and occurs at least twice. This filters fragments like `of the` while keeping `notice period` and `shall not solicit`.

**A cost of lemmatization to discuss in your report:** fixed legal phrases get lemmatized too, so `governing law` becomes `govern law` and `registered office` becomes `register office`. The phrase is still findable (queries are lemmatized the same way), but it is less readable, and a strict phrase-matching system would treat it as a different term. Keeping surface forms for known multi-word legal terms is a sensible Stage 2 refinement.

In [57]:
from nltk.util import ngrams as nltk_ngrams

FUNCTION_WORDS = NLTK_STOP | ARCHAIC
def is_meaningful(gram):
    return gram[0] not in FUNCTION_WORDS and gram[-1] not in FUNCTION_WORDS

def ngram_counter(sent_docs, n):
    """sent_docs: {doc_id: [[tokens of sentence 1], ...]}. N-grams never cross a sentence boundary."""
    c = Counter()
    for sents in sent_docs.values():
        for sent in sents:
            c.update(nltk_ngrams(sent, n))
    return c

final_tokens = processed_sents["Final Pipeline"]
table_f, ngram_counters = [], {}
names = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "4-Gram", 5: "5-Gram"}
for n, label in names.items():
    c = ngram_counter(final_tokens, n); ngram_counters[n] = c
    meaningful = {g: f for g, f in c.items() if f >= 2 and is_meaningful(g)}
    table_f.append({"N-Gram": label, "Total": sum(c.values()), "Unique": len(c),
                    "Meaningful (freq>=2)": len(meaningful),
                    "Top N-Grams": "; ".join(f"{' '.join(g)} ({f})" for g, f in c.most_common(10))})
table_f = pd.DataFrame(table_f)
save(table_f, "ngram_results.csv")

for n, fname in [(1, "unigram_results.csv"), (2, "bigram_results.csv"), (3, "trigram_results.csv")]:
    save(pd.DataFrame([{"ngram": " ".join(g), "frequency": f, "meaningful": is_meaningful(g)}
                       for g, f in ngram_counters[n].most_common()]), fname)

term_dict = len(term_dictionary(after_tokens))
print(f"\nTerm-based dictionary (unique unigram terms): {term_dict:,}")
for n in range(2, 6):
    print(f"  {names[n]:8s} dictionary: {len(ngram_counters[n]):,}  ({len(ngram_counters[n]) / term_dict:.1f}x the term dictionary)")
table_f

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/ngram_results.csv
  saved -> /content/drive/MyDrive/NLP_Assessment1/results/unigram_results.csv
  saved -> /content/drive/MyDrive/NLP_Assessment1/results/bigram_results.csv
  saved -> /content/drive/MyDrive/NLP_Assessment1/results/trigram_results.csv

Term-based dictionary (unique unigram terms): 15,375
  Bigram   dictionary: 75,017  (4.9x the term dictionary)
  Trigram  dictionary: 100,251  (6.5x the term dictionary)
  4-Gram   dictionary: 104,050  (6.8x the term dictionary)
  5-Gram   dictionary: 101,844  (6.6x the term dictionary)


,N-Gram,Total,Unique,Meaningful (freq>=2),Top N-Grams
0,Unigram,178427,15407,6521,any (3534); company (3280); will (2382); shall (2280); agreement (1549); not (1464); employee (1282); may (1131); pa...
1,Bigram,168425,75017,19317,confidential information (364); e e (330); company 's (328); term condition (308); shall not (287); time time (264);...
2,Trigram,158788,100251,16787,e e e (152); rajarajeswari college engineering (146); include not limit (91); 14 ramohalli cross (74); any confident...
3,4-Gram,149453,104050,13909,e e e e (65); engineering rajarajeswari college engineering (62); department electrical electronics engineering (55)...
4,5-Gram,140440,101844,11918,department electrical electronics engineering rajarajeswari (50); ramohalli cross kumbalgodu mysore road (48); elect...


In [58]:
print("Top 10 MEANINGFUL n-grams (the domain phrases worth indexing):\n")
for n in range(2, 6):
    top = [(" ".join(g), f) for g, f in ngram_counters[n].most_common() if f >= 2 and is_meaningful(g)][:10]
    print(f"{names[n]}:")
    for g, f in top:
        print(f"   {f:4d}  {g}")

Top 10 MEANINGFUL n-grams (the domain phrases worth indexing):

Bigram:
    364  confidential information
    330  e e
    328  company 's
    308  term condition
    264  time time
    205  intellectual property
    195  employment company
    193  college engineering
    182  company shall
    162  rajarajeswari college
Trigram:
    152  e e e
    146  rajarajeswari college engineering
     91  include not limit
     74  14 ramohalli cross
     73  visvesvaraya technological university
     71  company reserve right
     68  term condition employment
     66  engineering rajarajeswari college
     62  electrical electronics engineering
     61  intellectual property right
4-Gram:
     65  e e e e
     62  engineering rajarajeswari college engineering
     55  department electrical electronics engineering
     50  cross kumbalgodu mysore road
     50  electrical electronics engineering rajarajeswari
     50  electronics engineering rajarajeswari college
     49  14 ramohalli cross kum

---
# Module 3: Process Selection and Ordering

### Justification of every process

| Process | Required? | Placement | Depends on | What happens if moved |
|---|---|---|---|---|
| **Text extraction (with OCR)** | Yes | First | Nothing | Nothing can precede it. Scanned pages are OCR'd here, page by page, before any other step sees them. OCR errors (a dropped rupee sign, lost spaces in headings) then propagate into every later step and metric. |
| **Cleaning** | Yes | Before tokenization | Extraction | After tokenization, page headers and enumeration labels are already scattered tokens and much harder to find. |
| **Number normalisation** | Yes (domain) | After cleaning, before tokenization | Cleaning | After tokenization, `6 (six) months` is already split into four tokens, and the pattern is lost. |
| **Tokenization (hybrid)** | Yes | After normalisation | Clean text | Everything downstream counts tokens; a bad tokenizer propagates to POS, NER, the index and every metric. |
| **NER** | Yes (domain) | On *cased*, unlowercased text | Tokenization | After lowercasing, `Acme Technologies` and `Bengaluru` lose the capitalisation cue and recall drops sharply. |
| **POS tagging** | Yes | Before lemmatization | Tokenization | spaCy's lemmatizer *uses* the POS tag (`binding` as ADJ vs VERB). Tagging after would give worse lemmas. |
| **Lemmatization** | Yes | After POS, before stop-word removal | POS tags | Before POS it cannot disambiguate. Lemmatizing before stop removal lets inflected stop words (`was` to `be`) be caught by the list. |
| **Stemming** | No (Pipeline A only) | After stop-word removal | Stop list | Before stop removal it rewrites stop words (`this` to `thi`) so they escape the list; Exercise 9 measures this. |
| **Lowercasing** | Yes | After NER and lemmatization | Lemmas | Before NER it destroys capitalisation cues. |
| **Stop-word removal (custom)** | Yes, with a custom list | After lemmatization, before n-grams and indexing | Lemmas | With a standard list, negations vanish and meaning flips (Exercise 7). Skipping it entirely (Pipeline B) inflates the index and fills n-grams with `of the`. |
| **N-grams** | Yes | After stop-word removal | Final token stream | Before stop removal the top n-grams are function-word pairs; generated across sentences they fuse unrelated clauses. |
| **BPE** | Analysis only | Parallel branch | Clean text | Used as the *index* vocabulary it would split `termination` into `termin` + `ation`, making Boolean search on whole terms impossible. |
| **Inverted index** | Yes | After final tokens | Every step above | The index stores whatever the pipeline produces, so every earlier choice is baked in. |
| **Query processing** | Yes | At search time | Same pipeline as the index | Queries processed differently from documents never match (e.g. an unstemmed query against a stemmed index). |

### Mandatory experiment: Pipeline A vs Pipeline B vs Final
The comparison below uses the pipelines defined before Exercise 11. Pipeline A and B are the two branches of the assignment diagram; the Final pipeline is the one this design argues for.

In [59]:
DOMAIN_PROBES = ["employee", "employer", "employment", "confidential", "confidence", "terminal", "termination",
                 "non-compete", "non-solicitation", "shall not", "without notice", "₹12,00,000",
                 "Section 27", "notice period"]

def domain_terms_preserved(fn):
    """A probe is preserved if the pipeline keeps it (non-empty) AND it does not collide with another probe."""
    outs = {p: tuple(fn(p)) for p in DOMAIN_PROBES}
    counts = Counter(outs.values())
    return sum(1 for p, o in outs.items() if o and counts[o] == 1), outs

def negation_kept(fn):
    tests = ["The Employee shall not disclose.", "The Employer may terminate without notice.",
             "The Employee must never solicit clients."]
    return sum(1 for t in tests if any(w in fn(t) for w in ["not", "without", "never"])), len(tests)

def meaningful_ngrams(sent_docs):
    total = 0
    for n in (2, 3):
        c = ngram_counter(sent_docs, n)
        total += sum(1 for g, f in c.items() if f >= 2 and is_meaningful(g))
    return total

comparison_rows, probe_outputs = [], {}
for name, fn in PIPELINES.items():
    toks = [t for d in processed[name].values() for t in d]
    kept, outs = domain_terms_preserved(fn)
    neg, neg_total = negation_kept(fn)
    probe_outputs[name] = outs
    comparison_rows.append({"Pipeline": name, "Token Count": len(toks), "Vocabulary Size": len(set(toks)),
                            "Meaningful N-Grams": meaningful_ngrams(processed_sents[name]),
                            "Domain Terms Preserved": f"{kept}/{len(DOMAIN_PROBES)}",
                            "Negations Preserved": f"{neg}/{neg_total}",
                            "Preprocessing Time (s)": round(pipe_time[name], 2)})
pipe_structural = pd.DataFrame(comparison_rows)
display(pipe_structural)

print("How each pipeline represents the domain probes (collisions show as identical outputs):")
pd.DataFrame(probe_outputs).map(lambda t: " ".join(t) if t else "(removed)")

,Pipeline,Token Count,Vocabulary Size,Meaningful N-Grams,Domain Terms Preserved,Negations Preserved,Preprocessing Time (s)
0,Pipeline A,169729,13838,38929,10/14,2/3,3.31
1,Pipeline B,290420,15054,21447,14/14,3/3,58.99
2,Final Pipeline,178427,15407,36104,14/14,3/3,53.79


How each pipeline represents the domain probes (collisions show as identical outputs):


,Pipeline A,Pipeline B,Final Pipeline
employee,employe,employee,employee
employer,employ,employer,employer
employment,employ,employment,employment
confidential,confidenti,confidential,confidential
confidence,confid,confidence,confidence
terminal,termin,terminal,terminal
termination,termin,termination,termination
non-compete,non-compet,non compete,non-compete
non-solicitation,non-solicit,non solicitation,non-solicitation
shall not,shall,shall not,shall not


---
# Module 4: Information Retrieval

A **positional** inverted index is built for each pipeline: `term -> {doc_id: [positions]}`. Positions are what make phrase search possible; a plain posting list (`term -> [doc_ids]`) can answer Boolean queries but not "is `notice` immediately followed by `period`?".

**Query processing rule:** every query operand is passed through *the same pipeline as the documents*. A multi-word operand is treated as a phrase. Operator precedence is `NOT` > `AND` > `OR`.

**A problem found while building this, and its fix.** "The same pipeline" is not enough for lemmatization. spaCy's lemmatizer depends on context. In a document, `towards training costs` makes `training` a noun, so its lemma is `training`. A query operand `training` has no context, so spaCy tags it as a verb and lemmatizes it to `train`. The index holds `training`, the query asks for `train`, and nothing matches. Stemming never has this problem because it is context-free, which is why a naive implementation makes Pipeline A look *better* at retrieval than it really is.

**Fix: corpus-consistent query normalisation.** While indexing, record which lemma each surface form actually received in the corpus. At query time, map each query word to that corpus lemma (falling back to its isolated lemma if the word never occurs). Module 6 measures this fix as an ablation.

In [60]:
def build_index(token_docs):
    index = defaultdict(lambda: defaultdict(list))
    for did, toks in token_docs.items():
        for pos, tok in enumerate(toks):
            index[tok][did].append(pos)
    return {t: dict(p) for t, p in index.items()}

INDEXES = {name: build_index(processed[name]) for name in PIPELINES}
ALL_DOCS = set(docs)

# Surface form -> the lemma the corpus most often gave it, per lemmatizing pipeline.
surf_B, surf_F = defaultdict(Counter), defaultdict(Counter)
for t in docs.values():
    for tok in nlp(clean_text(t)):
        if not tok.is_space and not tok.is_punct:
            surf_B[tok.text.lower()][tok.lemma_.lower()] += 1
    for tok in final_doc(t):
        if not tok.is_space and not tok.is_punct:
            surf_F[tok.text.lower()][final_token(tok)] += 1
CORPUS_LEMMA = {"Pipeline B": {k: c.most_common(1)[0][0] for k, c in surf_B.items()},
                "Final Pipeline": {k: c.most_common(1)[0][0] for k, c in surf_F.items()}}

def query_B(text):
    return [CORPUS_LEMMA["Pipeline B"].get(t.text.lower(), t.lemma_.lower())
            for t in nlp(clean_text(text)) if not t.is_space and not t.is_punct]

def query_final(text):
    out = []
    for t in final_doc(text):
        if t.is_space or t.is_punct: continue
        tok = CORPUS_LEMMA["Final Pipeline"].get(t.text.lower(), final_token(t))
        if tok not in CUSTOM_STOP: out.append(tok)
    return out

CONFIG.setdefault("CORPUS_CONSISTENT_QUERIES", True)
QUERY_FN_CONSISTENT = {"Pipeline A": pipeline_A, "Pipeline B": query_B, "Final Pipeline": query_final}
QUERY_FN_NAIVE = dict(PIPELINES)   # plain pipeline on the isolated query: the version with the bug
QUERY_FN = QUERY_FN_CONSISTENT if CONFIG["CORPUS_CONSISTENT_QUERIES"] else QUERY_FN_NAIVE
print("Query 'training' -> naive:", QUERY_FN_NAIVE["Final Pipeline"]("training"),
      "| corpus-consistent:", QUERY_FN_CONSISTENT["Final Pipeline"]("training"))

final_index = INDEXES["Final Pipeline"]
with open(os.path.join(CONFIG["RESULTS_DIR"], "inverted_index.json"), "w", encoding="utf-8") as f:
    json.dump({t: {"df": len(p), "postings": p} for t, p in sorted(final_index.items())}, f, ensure_ascii=False, indent=1)
print(f"  saved -> {CONFIG['RESULTS_DIR']}/inverted_index.json")

for name, idx in INDEXES.items():
    print(f"{name:15s} index terms: {len(idx):,}")
print("\nSample posting lists (Final Pipeline):")
for term in ["employee", "non-compete", "confidential", "notice", "not"]:
    if term in final_index:
        print(f"  {term:14s} -> {', '.join(sorted(final_index[term]))}")

Query 'training' -> naive: ['train'] | corpus-consistent: ['training']
  saved -> /content/drive/MyDrive/NLP_Assessment1/results/inverted_index.json
Pipeline A      index terms: 13,838
Pipeline B      index terms: 15,054
Final Pipeline  index terms: 15,407

Sample posting lists (Final Pipeline):
  employee       -> D13, D14, D15, D16, D17, D18, D19, D20, D21, D22, D23, D24, D25, D27, D28, D29, D30, D32, D33, D34, D35, D36, D37, D38, D39, D41, D42, D43, D44
  non-compete    -> D13, D16, D18, D19, D32, D34, D35, D36
  confidential   -> D13, D14, D15, D16, D17, D18, D19, D20, D21, D22, D23, D24, D25, D27, D28, D29, D30, D32, D34, D35, D36, D41, D42, D43, D44
  notice         -> D13, D15, D16, D18, D19, D20, D21, D22, D24, D25, D27, D29, D30, D32, D34, D37, D38, D39, D40, D41, D42, D43, D44
  not            -> D01, D02, D03, D04, D05, D06, D07, D08, D09, D10, D11, D12, D13, D14, D15, D16, D17, D18, D19, D20, D21, D22, D23, D24, D25, D26, D27, D28, D29, D30, D32, D33, D34, D35, D37, D38, D3

In [61]:
def phrase_docs(index, terms):
    """Documents where `terms` occur consecutively."""
    if not terms: return set()
    if len(terms) == 1: return set(index.get(terms[0], {}))
    candidates = set.intersection(*(set(index.get(t, {})) for t in terms))
    hits = set()
    for did in candidates:
        starts = set(index[terms[0]][did])
        for k, t in enumerate(terms[1:], start=1):
            starts &= {p - k for p in index[t][did]}
            if not starts: break
        if starts: hits.add(did)
    return hits

def tokenize_query(q):
    """Split into operands (quoted phrases or word runs) and operators AND/OR/NOT."""
    parts = re.findall(r'"[^"]+"|\bAND\b|\bOR\b|\bNOT\b|[^\s"]+', q)
    out, buf = [], []
    for p in parts:
        if p in {"AND", "OR", "NOT"}:
            if buf: out.append(("TERM", " ".join(buf))); buf = []
            out.append(("OP", p))
        elif p.startswith('"'):
            if buf: out.append(("TERM", " ".join(buf))); buf = []
            out.append(("TERM", p.strip('"')))
        else:
            buf.append(p)
    if buf: out.append(("TERM", " ".join(buf)))
    return out

def evaluate_query(q, pipeline_name, operand_fn=None, qfns=None):
    """Boolean evaluation with precedence NOT > AND > OR. operand_fn(text) -> set of doc ids."""
    idx, pfn = INDEXES[pipeline_name], (qfns or QUERY_FN)[pipeline_name]
    operand_fn = operand_fn or (lambda text: phrase_docs(idx, pfn(text)))
    toks = tokenize_query(q)
    or_groups, and_group, negate, pending_and = [], None, False, False
    def push(s):
        nonlocal and_group, negate
        s = ALL_DOCS - s if negate else s
        negate = False
        and_group = s if and_group is None else and_group & s
    for kind, val in toks:
        if kind == "TERM":
            push(operand_fn(val))
        elif val == "NOT":
            negate = not negate
        elif val == "OR":
            if and_group is not None: or_groups.append(and_group)
            and_group = None
    if and_group is not None: or_groups.append(and_group)
    return set().union(*or_groups) if or_groups else set()

def query_type(q):
    if " NOT " in f" {q} ": return "Boolean NOT"
    if " AND " in f" {q} ": return "Boolean AND"
    if " OR " in f" {q} ": return "Boolean OR"
    n = len(q.strip('"').split())
    return {1: "Unigram", 2: "Bigram", 3: "Trigram"}.get(n, f"{n}-gram phrase")

def rank(docs_found, q, pipeline_name, qfns=None):
    """Order retrieved documents by summed log-tf x idf of the positive query terms (for P@K / R@K)."""
    idx, pfn, N = INDEXES[pipeline_name], (qfns or QUERY_FN)[pipeline_name], len(ALL_DOCS)
    terms = [t for kind, v in tokenize_query(q) if kind == "TERM" for t in pfn(v)]
    def score(did):
        s = 0.0
        for t in terms:
            post = idx.get(t, {})
            if did in post:
                s += (1 + math.log(len(post[did]))) * math.log(N / len(post))
        return s
    return sorted(docs_found, key=lambda d: (-score(d), d))

def search(q, pipeline_name="Final Pipeline", qfns=None):
    t0 = time.perf_counter()
    found = evaluate_query(q, pipeline_name, qfns=qfns)
    ranked = rank(found, q, pipeline_name, qfns=qfns)
    return ranked, (time.perf_counter() - t0) * 1000

for q in ["non-compete", "notice period", "probation AND termination", "stipend OR remuneration",
          "confidential AND NOT non-compete"]:
    r, ms = search(q)
    print(f"{query_type(q):12s} {q!r:40s} -> {len(r):2d} docs {r[:8]} ({ms:.2f} ms)")

Unigram      'non-compete'                            ->  8 docs ['D13', 'D18', 'D36', 'D35', 'D16', 'D19', 'D32', 'D34'] (12.81 ms)
Bigram       'notice period'                          ->  9 docs ['D13', 'D16', 'D25', 'D19', 'D32', 'D18', 'D42', 'D38'] (10.08 ms)
Boolean AND  'probation AND termination'              ->  7 docs ['D13', 'D16', 'D19', 'D18', 'D32', 'D38', 'D29'] (17.96 ms)
Boolean OR   'stipend OR remuneration'                -> 12 docs ['D13', 'D25', 'D19', 'D41', 'D33', 'D18', 'D16', 'D32'] (17.73 ms)
Boolean NOT  'confidential AND NOT non-compete'       -> 17 docs ['D25', 'D14', 'D23', 'D42', 'D20', 'D24', 'D21', 'D15'] (18.00 ms)


---
# Module 5: Domain-specific query analysis

Fifteen queries covering every required type. They are written for employment contracts; **replace or extend them** to match what your documents actually contain.

In [62]:
QUERIES = [  # YOUR INPUT: adapt to your corpus
    ("Q01", "stipend"), ("Q02", "non-compete"), ("Q03", "gratuity"),
    ("Q04", "notice period"), ("Q05", "confidential information"), ("Q06", "intellectual property"),
    ("Q07", "governing law"),
    ("Q08", "shall not disclose"), ("Q09", "shall not solicit"), ("Q10", "work from home"),
    ("Q11", "probation AND termination"), ("Q12", "bond AND training"),
    ("Q13", "stipend OR remuneration"), ("Q14", "confidential OR proprietary"),
    ("Q15", "confidential AND NOT non-compete"),
]

retrieval_rows = []
for qid, q in QUERIES:
    for name in PIPELINES:
        ranked, ms = search(q, name)
        retrieval_rows.append({"Query ID": qid, "Query": q, "Type": query_type(q), "Pipeline": name,
                               "Retrieved Documents": " ".join(ranked), "# Results": len(ranked),
                               "Time (ms)": round(ms, 3)})
retrieval_df = pd.DataFrame(retrieval_rows)
save(retrieval_df, "retrieval_results.csv")
retrieval_df[retrieval_df.Pipeline == "Final Pipeline"].drop(columns="Pipeline")

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/retrieval_results.csv


,Query ID,Query,Type,Retrieved Documents,# Results,Time (ms)
2,Q01,stipend,Unigram,D13 D25 D19 D33,4,9.397
5,Q02,non-compete,Unigram,D13 D18 D36 D35 D16 D19 D32 D34,8,9.570
8,Q03,gratuity,Unigram,D13 D19 D18 D16 D32,5,9.075
11,Q04,notice period,Bigram,D13 D16 D25 D19 D32 D18 D42 D38 D22,9,9.376
14,Q05,confidential information,Bigram,D13 D19 D25 D18 D16 D14 D23 D42 D32 D20 D24 D15 D44 D21 D29 D27 D22 D36 D28,19,10.606
17,Q06,intellectual property,Bigram,D13 D32 D25 D16 D19 D18 D27 D15 D21 D29 D44 D42 D22 D30 D43 D26,16,9.177
20,Q07,governing law,Bigram,D13 D25 D19 D16 D32 D15 D42 D30 D26 D34 D27 D36 D28 D43 D21 D35,16,9.290
23,Q08,shall not disclose,Trigram,D41 D18 D29 D21 D33,5,10.132
26,Q09,shall not solicit,Trigram,D25 D28 D35,3,10.155
29,Q10,work from home,Trigram,D17 D37 D19,3,9.801


### Negation demonstration: why Pipeline A cannot be trusted for this domain

The two queries below mean opposite things. A pipeline that removes `not` treats them as the same query and returns the same documents for both.

In [63]:
for q in ["shall not disclose", "shall disclose"]:
    print(f"Query: {q!r}")
    for name in PIPELINES:
        r, _ = search(q, name)
        print(f"   {name:15s} processed as {QUERY_FN[name](q)!s:36s} -> {len(r):2d} docs")
    print()

Query: 'shall not disclose'
   Pipeline A      processed as ['shall', 'disclos']                 ->  8 docs
   Pipeline B      processed as ['shall', 'not', 'disclose']         ->  5 docs
   Final Pipeline  processed as ['shall', 'not', 'disclose']         ->  5 docs

Query: 'shall disclose'
   Pipeline A      processed as ['shall', 'disclos']                 ->  8 docs
   Pipeline B      processed as ['shall', 'disclose']                ->  3 docs
   Final Pipeline  processed as ['shall', 'disclose']                ->  3 docs



---
# Module 6: Retrieval evaluation

A **relevance set** records, for each query, which documents are genuinely relevant. It must be made by reading the documents, and ideally *before* looking at what the system retrieves, otherwise the evaluation is biased toward the system.

> **Why the provisional set flatters Pipeline A.** Provisional relevance is literal text matching, which is roughly what a stemming pipeline does anyway, and it has no notion of meaning. It counts a document as relevant to `shall disclose` only if that exact string appears, so it cannot reward a pipeline for correctly rejecting `shall not disclose`. Real relevance judgements made by reading the contracts will penalise Pipeline A on exactly the negation queries where it is wrong. Treat any provisional ranking of the pipelines as meaningless.

> **YOUR INPUT (required for valid numbers).** The notebook writes `annotations/relevance.csv` the first time it runs. Until you fill it, the notebook uses a **provisional** relevance set built by literal text matching on the raw documents, so every cell runs. Provisional results are stamped as such and **must not be reported**. Fill the `relevant_docs` column with space-separated IDs (e.g. `D01 D04 D07`), then re-run this module.

In [64]:
REL_PATH = os.path.join(CONFIG["ANNOTATION_DIR"], "relevance.csv")
raw_lower = {did: re.sub(r"\s+", " ", t.lower()) for did, t in docs.items()}

def provisional_relevance(q):
    """Literal substring evaluation on raw text. A rough stand-in, NOT a real relevance judgement."""
    return evaluate_query(q, "Final Pipeline",
                          operand_fn=lambda text: {d for d, t in raw_lower.items() if text.lower() in t})

if not os.path.exists(REL_PATH):
    pd.DataFrame([{"query_id": qid, "query": q, "relevant_docs": ""} for qid, q in QUERIES]).to_csv(REL_PATH, index=False)
    print(f"Created a blank relevance template at {REL_PATH}. Fill it in and re-run this module.")

rel_df = pd.read_csv(REL_PATH, dtype=str).fillna("")
filled = rel_df["relevant_docs"].str.strip().astype(bool).any()
RELEVANCE, REL_SOURCE = {}, {}
for qid, q in QUERIES:
    row = rel_df[rel_df.query_id == qid]
    if filled and len(row) and row.iloc[0]["relevant_docs"].strip():
        RELEVANCE[qid] = set(row.iloc[0]["relevant_docs"].split()); REL_SOURCE[qid] = "manual"
    else:
        RELEVANCE[qid] = provisional_relevance(q); REL_SOURCE[qid] = "PROVISIONAL"

n_prov = sum(v == "PROVISIONAL" for v in REL_SOURCE.values())
if n_prov:
    print(f"!! {n_prov} of {len(QUERIES)} queries use PROVISIONAL relevance. Do not report these numbers.")
else:
    print("All queries use your manual relevance judgements.")

Created a blank relevance template at /content/drive/MyDrive/NLP_Assessment1/annotations/relevance.csv. Fill it in and re-run this module.
!! 15 of 15 queries use PROVISIONAL relevance. Do not report these numbers.


In [65]:
K = CONFIG["TOP_K"]
def prf(ret, rel):
    ret_s = set(ret); tp = len(ret_s & rel)
    p = tp / len(ret_s) if ret_s else (1.0 if not rel else 0.0)
    r = tp / len(rel) if rel else np.nan
    f = np.nan if np.isnan(r) else (0.0 if p + r == 0 else 2 * p * r / (p + r))
    topk = ret[:K]; tpk = len(set(topk) & rel)
    pk = tpk / len(topk) if topk else (1.0 if not rel else 0.0)
    rk = tpk / len(rel) if rel else np.nan
    return p, r, f, pk, rk

eval_rows = []
for qid, q in QUERIES:
    rel = RELEVANCE[qid]
    for name in PIPELINES:
        ranked, ms = search(q, name)
        p, r, f, pk, rk = prf(ranked, rel)
        eval_rows.append({"Query ID": qid, "Query": q, "Type": query_type(q), "Pipeline": name,
                          "Relevant": len(rel), "Retrieved": len(ranked), "Precision": p, "Recall": r,
                          "F1": f, f"P@{K}": pk, f"R@{K}": rk, "Time (ms)": ms,
                          "Relevance Source": REL_SOURCE[qid]})
eval_df = pd.DataFrame(eval_rows)
save(eval_df.round(3), "evaluation_results.csv")

per_type = (eval_df[eval_df.Pipeline == "Final Pipeline"]
            .groupby("Type")[["Precision", "Recall", "F1", f"P@{K}", f"R@{K}"]].mean().round(3))
print("Final pipeline, averaged by query type (NaN = no relevant documents exist for that query):")
per_type

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/evaluation_results.csv
Final pipeline, averaged by query type (NaN = no relevant documents exist for that query):


,Precision,Recall,F1,P@5,R@5
Type,,,,,
Bigram,0.984,1.000,0.992,1.000,0.366
Boolean AND,1.000,0.771,0.867,1.000,0.646
Boolean NOT,1.000,1.000,1.000,1.000,0.294
Boolean OR,1.000,1.000,1.000,1.000,0.308
Trigram,0.889,1.000,0.933,0.889,1.000
Unigram,1.000,1.000,1.000,1.000,0.875


### Ablation: naive vs corpus-consistent query lemmatization

The same final pipeline, evaluated twice: once processing queries naively (isolated lemmatization, the version with the context bug) and once with corpus-consistent normalisation. The difference is the measured value of the fix.

In [66]:
abl = []
for label, qfns in [("naive query lemmatization", QUERY_FN_NAIVE), ("corpus-consistent (used)", QUERY_FN_CONSISTENT)]:
    ps, rs, fs, changed = [], [], [], []
    for qid, q in QUERIES:
        ranked, _ = search(q, "Final Pipeline", qfns=qfns)
        p, r, f, _, _ = prf(ranked, RELEVANCE[qid])
        ps.append(p); rs.append(r); fs.append(f)
    abl.append({"Final pipeline, queries processed by": label, "Precision": np.nanmean(ps),
                "Recall": np.nanmean(rs), "F1": np.nanmean(fs)})
display(pd.DataFrame(abl).round(3))
diff = [(qid, q) for qid, q in QUERIES
        if search(q, "Final Pipeline", QUERY_FN_NAIVE)[0] != search(q, "Final Pipeline", QUERY_FN_CONSISTENT)[0]]
print("Queries whose results change with the fix:", diff if diff else "none")

,"Final pipeline, queries processed by",Precision,Recall,F1
0,naive query lemmatization,0.974,0.947,0.947
1,corpus-consistent (used),0.974,0.969,0.967


Queries whose results change with the fix: [('Q12', 'bond AND training')]


---
# Results summary (Tables H, I, J)

In [67]:
metrics = (eval_df.groupby("Pipeline")[["Precision", "Recall", "F1", "Time (ms)"]]
           .mean().reindex(list(PIPELINES)))

# Table H: pipeline comparison
table_h = pipe_structural.set_index("Pipeline").join(metrics[["Precision", "Recall", "F1"]].round(3))
table_h = table_h.T.reset_index().rename(columns={"index": "Measure"})
save(table_h, "pipeline_comparison.csv")
print("TABLE H: Pipeline comparison")
display(table_h)

# Table I: retrieval results, one representative query per type (Final Pipeline)
fin = retrieval_df[retrieval_df.Pipeline == "Final Pipeline"]
table_i = fin.groupby("Type").first().reset_index()[["Query", "Type", "Retrieved Documents", "# Results", "Time (ms)"]]
print("\nTABLE I: Retrieval results (Final Pipeline, first query of each type)")
display(table_i)

# Table J: overall performance
table_j = metrics.copy()
table_j["Preprocessing Time (s)"] = [round(pipe_time[p], 2) for p in table_j.index]
table_j = table_j.round(3).reset_index().rename(columns={"Pipeline": "Method/Pipeline",
                                                         "Time (ms)": "Avg Query Time (ms)"})
print("\nTABLE J: Overall performance")
display(table_j)
if n_prov:
    print(f"\n!! These tables use PROVISIONAL relevance for {n_prov} queries. Fill annotations/relevance.csv first.")

  saved -> /content/drive/MyDrive/NLP_Assessment1/results/pipeline_comparison.csv
TABLE H: Pipeline comparison


Pipeline,Measure,Pipeline A,Pipeline B,Final Pipeline
0,Token Count,169729,290420,178427
1,Vocabulary Size,13838,15054,15407
2,Meaningful N-Grams,38929,21447,36104
3,Domain Terms Preserved,10/14,14/14,14/14
4,Negations Preserved,2/3,3/3,3/3
5,Preprocessing Time (s),3.31,58.99,53.79
6,Precision,0.945,0.964,0.974
7,Recall,0.969,0.962,0.969
8,F1,0.95,0.957,0.967



TABLE I: Retrieval results (Final Pipeline, first query of each type)


,Query,Type,Retrieved Documents,# Results,Time (ms)
0,notice period,Bigram,D13 D16 D25 D19 D32 D18 D42 D38 D22,9,9.376
1,probation AND termination,Boolean AND,D13 D16 D19 D18 D32 D38 D29,7,17.688
2,confidential AND NOT non-compete,Boolean NOT,D25 D14 D23 D42 D20 D24 D21 D15 D29 D41 D44 D27 D17 D30 D22 D28 D43,17,17.426
3,stipend OR remuneration,Boolean OR,D13 D25 D19 D41 D33 D18 D16 D32 D27 D15 D22 D42,12,17.211
4,shall not disclose,Trigram,D41 D18 D29 D21 D33,5,10.132
5,stipend,Unigram,D13 D25 D19 D33,4,9.397



TABLE J: Overall performance


,Method/Pipeline,Precision,Recall,F1,Avg Query Time (ms),Preprocessing Time (s)
0,Pipeline A,0.945,0.969,0.950,0.480,3.31
1,Pipeline B,0.964,0.962,0.957,13.051,58.99
2,Final Pipeline,0.974,0.969,0.967,12.649,53.79



!! These tables use PROVISIONAL relevance for 15 queries. Fill annotations/relevance.csv first.


In [68]:
sel = table_j.set_index("Method/Pipeline")[["F1"]].copy()
frac = lambda v: int(str(v).split("/")[0])
sel["domain_terms"] = pipe_structural.set_index("Pipeline")["Domain Terms Preserved"].map(frac)
sel["negations"] = pipe_structural.set_index("Pipeline")["Negations Preserved"].map(frac)
sel["meaningful_ngrams"] = pipe_structural.set_index("Pipeline")["Meaningful N-Grams"]
# Rank by F1, then domain terms preserved, then negations preserved, then n-gram quality
# (all four are comparison criteria named in the assignment).
sel = sel.sort_values(["F1", "domain_terms", "negations", "meaningful_ngrams"], ascending=False, kind="stable")
best_name = sel.index[0]
tied = sel[sel.F1 == sel.F1.iloc[0]].index.tolist()
print(f"Selected pipeline: {best_name}")
if len(tied) > 1:
    print(f"  F1 is TIED between {tied} ({sel.F1.iloc[0]:.3f}); the tie is broken by domain terms preserved,")
    print("  then negations preserved, then meaningful n-grams. A tie usually means the relevance set is too easy or provisional:")
    print("  real, manual judgements on negation-sensitive queries should separate the pipelines.\n")
else:
    print(f"  Highest mean F1 ({sel.F1.iloc[0]:.3f}).\n")
display(sel)
print("Selection reasoning to adapt for your report:")
print(" - F1 is the primary criterion because it balances missing relevant contracts (recall)")
print("   against returning wrong ones (precision); both have a real cost to someone reviewing a contract.")
print(" - Pipeline A's smaller vocabulary looks efficient, but it merges distinct legal terms and drops")
print("   negations, so it answers 'shall disclose' and 'shall not disclose' identically (Module 5).")
print(" - Pipeline B keeps meaning but indexes every function word, which inflates the index and")
print("   fills the n-grams with fragments like 'of the'.")
print(" - The Final pipeline is designed to keep what matters (negation, modals, money, statutes,")
print("   party names) and drop what does not. If your real numbers disagree, report that honestly.")

Selected pipeline: Final Pipeline
  Highest mean F1 (0.967).



,F1,domain_terms,negations,meaningful_ngrams
Method/Pipeline,,,,
Final Pipeline,0.967,14,3,36104
Pipeline B,0.957,14,3,21447
Pipeline A,0.950,10,2,38929


Selection reasoning to adapt for your report:
 - F1 is the primary criterion because it balances missing relevant contracts (recall)
   against returning wrong ones (precision); both have a real cost to someone reviewing a contract.
 - Pipeline A's smaller vocabulary looks efficient, but it merges distinct legal terms and drops
   negations, so it answers 'shall disclose' and 'shall not disclose' identically (Module 5).
 - Pipeline B keeps meaning but indexes every function word, which inflates the index and
   fills the n-grams with fragments like 'of the'.
 - The Final pipeline is designed to keep what matters (negation, modals, money, statutes,
   party names) and drop what does not. If your real numbers disagree, report that honestly.


---
# What you must do before submitting

The notebook runs end to end on any folder of documents, but these items need **your** judgement, and the evaluation numbers are only valid once they are done.

1. **Documents.** Put 15 to 30 real employment or work contracts in `./data/`.
2. **OCR check (only if any of them are scans).** Install RapidOCR once (`pip install -U rapidocr onnxruntime pillow`). Read the OCR report right after Module 1 and compare at least one OCR'd page with its image, checking **every rupee amount**. RapidOCR often drops the rupee sign; the notebook restores only unmistakable cases and lists the ambiguous ones, so any amount without a sign needs your eye. Also check all-caps headings for lost spaces. Add a sentence to your report's limitations saying which documents were OCR'd and that OCR errors propagate into all later results.
3. **Relevance set (`annotations/relevance.csv`).** Required for Module 6. Read your documents and list the relevant IDs for each query, ideally before looking at the retrieval output.
4. **Queries (`QUERIES`, Module 5).** Adapt them to what your documents actually contain.
5. **Tokenizer gold (`TOKEN_GOLD`, Exercise 3).** Add at least five sentences from your own documents.
6. **POS gold (`POS_GOLD_TEST` and `POS_DOMAIN_TRAIN`, Exercise 12).** Verify every seed tag, add your own sentences, keep the two sets disjoint.
7. **NER gold (`NER_GOLD`, Exercise 13)** and the **`Correct?` column** in `results/ner_results.csv`.
8. **Write-ups.** Each exercise has a "what to write up" note. Rewrite them around *your* numbers; the reasoning transfers, the figures do not.
9. **GUI.** Not included in this notebook. It is a required component of the assignment, due with everything else.